In [4]:
! pip install -Uq oracledb sqlalchemy pandas sentence-transformers datasets einops "numpy<2.0"

# Part 1. Oracle AI Database Local Installation

## 1.1 Local Installation of Oracle AI Database

**Oracle AI Database 26ai** is a **converged database** built for AI developers.  
It unifies **relational, document, graph and vector data** in one engine — so you can build  
**semantic search**, **RAG**, and **natural language to SQL** applications without leaving the database.  

Store embeddings, run vector search, and apply AI directly where your data lives —  
**securely, efficiently, and at scale.**


For this notebook we will be using a local installation of [Oracle AI Database](https://www.oracle.com/database/free/get-started/)

1. Install & start Docker. Docker Desktop (Mac/Windows) or Docker Engine (Linux). Make sure it’s running.
    - If installed with Docker Enginer, run from terminal ```open /Applications/Docker.app```
2. Pull [docker image](https://www.oracle.com/database/free/get-started/)
3. Run a container with oracle image

    ```
    docker run -d \
      --name oracle-full \
      -p 1521:1521 -p 5500:5500 \
      -e ORACLE_PWD=OraclePwd_2025 \
      -e ORACLE_SID=FREE \
      -e ORACLE_PDB=FREEPDB1 \
      -v ~/oracle/full_data:/opt/oracle/oradata \
      container-registry.oracle.com/database/free:latest
    ```

> 🚫 **Troubleshoot**  
> If you see the error:  
> *`docker: Error response from daemon: Conflict. The container name "/oracle-full" is already in use by container ... You have to remove (or rename) that container to be able to reuse that name.`*  
>
> 🧩 **Fix:**  
> - Remove the existing container:  
>   ```bash
>   docker rm oracle-full
>   ```  
> - Then re-run your Docker command from **Step 3** to start a new container.


After running the docker command above in your terminal, you should see the image below if you click into the container running

![Docker Container Log](./images/docker_container_image.png)


### 1.1.1 Connecting to Oracle AI Database

In [2]:
import oracledb
import time

def connect_to_oracle(max_retries=3, retry_delay=5):
    """
    Connect to Oracle database with retry logic and better error handling.
    
    Args:
        max_retries: Maximum number of connection attempts
        retry_delay: Seconds to wait between retries
    """
    user = "system"
    password = "OraclePwd_2025"  # must match ORACLE_PWD from docker run
    dsn = "localhost:1521/FREEPDB1"
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Connection attempt {attempt}/{max_retries}...")
            conn = oracledb.connect(
                user=user,
                password=password,
                dsn=dsn
            )
            print("✓ Connected successfully!")
            
            # Test the connection
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE banner LIKE 'Oracle%';")
                banner = cur.fetchone()[0]
                print(f"\n{banner}")
            
            return conn
            
        except oracledb.OperationalError as e:
            error_msg = str(e)
            print(f"✗ Connection failed (attempt {attempt}/{max_retries})")
            
            if "DPY-4011" in error_msg or "Connection reset by peer" in error_msg:
                print("  → This usually means:")
                print("    1. Database is still starting up (wait 2-3 minutes)")
                print("    2. Listener is not bound to 0.0.0.0 (run fix_oracle_listener())")
                print("    3. Container is not running (check with check_docker_container())")
                
                if attempt < max_retries:
                    print(f"\n  Waiting {retry_delay} seconds before retry...")
                    time.sleep(retry_delay)
                else:
                    print("\n  💡 Try running:")
                    print("     1. check_docker_container() - verify container is running")
                    print("     2. fix_oracle_listener() - fix listener binding")
                    raise
            else:
                raise
        except Exception as e:
            print(f"✗ Unexpected error: {e}")
            raise
    
    raise ConnectionError("Failed to connect after all retries")

# Connect to Oracle
conn = connect_to_oracle()

Connection attempt 1/3...
✓ Connected successfully!

Oracle AI Database 26ai Free Release 23.26.1.0.0 - Develop, Learn, and Run for Free


> 🚫 Troubleshoot: Oracle Database Free (Docker) — Connection Fix
>
> If you see errors like:
>
> ```
> OperationalError: DPY-6005: cannot connect to database
> DPY-4011: the database or network closed the connection
> TNS-12545: Connect failed because target host or object does not exist
> ```
>
> It means the **Oracle listener** inside the container is binding to the **container hostname** instead of `0.0.0.0`, preventing host connections.
>
>
>
> 🧩 Fix
> 
> Run this exact command to patch the listener and restart it:
> 
> ```bash
> docker exec -it oracle-full bash -lc '
>   export ORACLE_HOME=${ORACLE_HOME:-/opt/oracle/product/26ai/dbhomeFree}
>   export PATH=$ORACLE_HOME/bin:$PATH
>   LISTENER_ORA="$ORACLE_HOME/network/admin/listener.ora"
> 
>   echo "== Fixing listener to use HOST=0.0.0.0"
>   sed -i "s/(HOST *= *[^)]*)/(HOST = 0.0.0.0)/" "$LISTENER_ORA"
> 
>   echo "== Restarting listener"
>   lsnrctl stop || true
>   lsnrctl start
> 
>   echo "== Re-registering services"
>   echo "ALTER SYSTEM REGISTER;" | sqlplus -s / as sysdba
> 
>   echo "== Listener status (first 20 lines):"
>   lsnrctl status | sed -n "1,20p"
> '
> ```
> 
> This:
> - Forces the listener to bind on all interfaces (`0.0.0.0`).
> - Restarts the listener.
> - Re-registers the PDB service (`FREEPDB1`) with the listener.
> 
> ---
> 


## 1.2 Remote Access with FreeSQL.com (Coming Soon)

---------

# Part 2. Data Loading, Preparation and Embedding Generation

## 2.1 Data Loading From Hugging Face

**Streaming the ArXiv Papers Dataset with Hugging Face**

The following code in the next cell demonstrates how to efficiently load and stream a large dataset using the **Hugging Face `datasets`** library — a powerful tool for handling massive datasets that may not fit into memory all at once.

```python
import pandas as pd
from datasets import load_dataset

ds_stream = load_dataset("nick007x/arxiv-papers", split="train", streaming=True)
```

**Step-by-Step Explanation**
1. **Importing dependencies**
   - `load_dataset` is imported from the `datasets` library, giving access to thousands of open datasets hosted on the Hugging Face Hub.

2. **Loading the dataset**
   - The dataset `"nick007x/arxiv-papers"` refers to a public dataset on the Hugging Face Hub that contains metadata or text from research papers hosted on [arXiv.org](https://arxiv.org).
   - The parameter `split="train"` loads the training partition of the dataset (many datasets have `train`, `validation`, and `test` splits).
   - The key argument `streaming=True` activates **streaming mode**, meaning the dataset is read progressively from the source without downloading it entirely to disk.

3. **Why streaming mode matters**
   - Traditional dataset loading downloads the full dataset into memory or disk, which can be slow and memory-intensive.  
   - Streaming allows you to process examples **as they arrive**, ideal for very large datasets or limited-resource environments.
   - You can iterate over the dataset like this:
     ```python
     for record in ds_stream:
         print(record)
         break
     ```

4. **Resulting object**
   - The variable `ds_stream` is an instance of `datasets.IterableDataset`, not a static table.  
   - You can convert a small sample into a Pandas DataFrame for inspection:
     ```python
     sample = [next(iter(ds_stream)) for _ in range(5)]
     pd.DataFrame(sample)
     ```

> **💡 Takeaway:**  
> Using `load_dataset(..., streaming=True)` enables developers and data scientists to work with **large datasets efficiently** — a perfect fit for machine learning pipelines, LLM training, or large document analysis workflows.


In [3]:
import pandas as pd
from datasets import load_dataset

In [ ]:
ds_stream = load_dataset("nick007x/arxiv-papers", split="train", streaming=True)

The code below streams the first 1,000 ArXiv papers(feel free to use more) from the dataset, extracts their titles and abstracts, combines them into a single text field, and stores the results as structured dictionaries for later use.

In [7]:
sampled = []
for i, item in enumerate(ds_stream):
    if i >= 1000:
        break
    
    arxiv_id = item.get("arxiv_id", f"unknown_{i}")
    title = item.get("title", "").strip()
    abstract = item.get("abstract", "").strip()
    
    # Combine title + abstract for embedding text
    text = f"{title} — {abstract}"
    
    sampled.append({
        "id": item.get("id", f"arxiv_{i}"),
        "arxiv_id": arxiv_id,
        "title": title,
        "abstract": abstract,
        "text": text
    })

print(f"✅ Streamed {len(sampled)} papers.")

✅ Streamed 1000 papers.


The code below converts the collected list of sampled paper records into a Pandas DataFrame for easier analysis, prints how many rows were loaded, and displays the first few entries to preview the dataset’s structure.

In [8]:
# Convert the list of tuples (id, text) into a DataFrame
dataset_df = pd.DataFrame(sampled)

# View shape and head
print(f"✅ Loaded {len(dataset_df)} rows into DataFrame.")
dataset_df.head()

✅ Loaded 1000 rows into DataFrame.


,id,arxiv_id,title,abstract,text
0,arxiv_0,0902.3253,The gravitational wave background from star-ma...,Stars on eccentric orbits around a massive bla...,The gravitational wave background from star-ma...
1,arxiv_1,0902.0428,Dynamics of planets in retrograde mean motion ...,"In a previous paper (Gayon &amp; Bois 2008a), ...",Dynamics of planets in retrograde mean motion ...
2,arxiv_2,0901.3401,Diurnal Thermal Tides in a Non-synchronized Ho...,We perform a linear analysis to investigate th...,Diurnal Thermal Tides in a Non-synchronized Ho...
3,arxiv_3,0901.1570,"Intermittent turbulence, noisy fluctuations an...",Recent research has shown that distinct physic...,"Intermittent turbulence, noisy fluctuations an..."
4,arxiv_4,0901.2048,Falling Transiting Extrasolar Giant Planets,We revisit the tidal stability of extrasolar s...,Falling Transiting Extrasolar Giant Planets — ...


## 2.2 Embedding Generation


This code in the next cell below imports the **`SentenceTransformer`** class from the `sentence_transformers` library and loads a pretrained model called **`"nomic-ai/nomic-embed-text-v1.5"`** from the Hugging Face Hub.

```python
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
```

🔍 What it does
- **`SentenceTransformer`** extends transformer-based models (like BERT or RoBERTa) to produce **sentence-level embeddings** — dense numerical vectors that capture the semantic meaning of text.
- The model **`nomic-ai/nomic-embed-text-v1.5`** is optimized for general-purpose text embeddings and works well for tasks such as:
  - Semantic search  
  - Clustering and topic modeling  
  - Retrieval-Augmented Generation (RAG)  
  - Similarity ranking and recommendation systems
- The parameter **`trust_remote_code=True`** allows the loader to execute custom code from the model’s repository, which is required for models that define specialized architectures or preprocessing logic.

In short, this code prepares a powerful embedding model that can transform text (like paper titles or abstracts) into **high-dimensional semantic vectors**, making it easier to measure meaning-based similarity across documents.


In [4]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

<All keys matched successfully>


This code in the next cell below iterates through each document, encodes it into a normalized embedding vector while showing live progress, and prefixes each text with "search_document: " so that the Nomic embedding model correctly interprets it as a retrieval document, improving alignment with query embeddings during semantic search.

The prefix `search_document`: " tells the Nomic embedding model what kind of text it’s encoding — in this case, a document intended for retrieval.

Nomic models like nomic-embed-text-v1.5 are trained with instructional prefixes (e.g., "search_query:", "search_document:", "classification:"), which guide the model to generate embeddings suited for different purposes.

**Why Use the `"search_document:"` Prefix with Nomic Embeddings**

Nomic embedding models (like **`nomic-embed-text-v1.5`**) are **instruction-tuned**, meaning they were trained with specific **task prefixes** that tell the model *how* to interpret the text you’re embedding.  

According to the [Nomic documentation](https://docs.nomic.ai/reference/api/embed-text-v-1-embedding-text-post) and the [Hugging Face model card](https://huggingface.co/nomic-ai/nomic-embed-text-v1.5):

> “Important: the text prompt must include a task instruction prefix, instructing the model which task is being performed.
> For example, if you are implementing a RAG application, you embed your documents as  
> `search_document: <text>` and embed your user queries as `search_query: <text>`.”

**💡 Why this matters**
- The prefix tells the model whether a text is a **document** or a **query**, ensuring both are embedded into the **same semantic space**.
- Using `search_document:` for document embeddings and `search_query:` for query embeddings **improves retrieval accuracy**, since the model optimizes for similarity between matching query–document pairs.
- Omitting or mismatching prefixes can lead to weaker alignment and lower recall in search or RAG workflows.


In [5]:
import numpy as np
import sys

In [ ]:
# Prefix for retrieval-style embeddings
dataset_df["text_prefixed"] = dataset_df["text"].apply(lambda x: f"search_document: {x}")

# Convert to list for iteration
texts = dataset_df["text_prefixed"].tolist()
embs = []

# Encode one text at a time with a single-line progress display
total = len(texts)
for i, text in enumerate(texts, start=1):
    embedding = embedding_model.encode(
        text,
        show_progress_bar=False,   # Disable SentenceTransformer progress
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    embs.append(embedding)

    # Print progress on the same line
    sys.stdout.write(f"\rEncoding {i}/{total} texts...")
    sys.stdout.flush()

# Final newline to avoid overwriting the last line
print(f"\n✅ Finished encoding {len(embs)} texts.")

# Convert list of vectors to NumPy array
embs = np.vstack(embs)
print(f"✅ Embeddings shape: {embs.shape}")


Encoding 1000/1000 texts...
✅ Finished encoding 1000 texts.
✅ Embeddings shape: (1000, 768)


One important detail to note is the embedding dimensionality — the number of numerical features in each vector representation.
This dimensionality determines the structure of your vector index and must remain consistent across all embeddings to ensure efficient similarity search and accurate retrieval performance.

This code in the next cell below converts each embedding into a list of 32-bit floating-point numbers (float32) so it can be stored in an Oracle VECTOR column, determines the embedding dimension (needed for defining the vector index), and we then displays the first two rows to confirm the data and embeddings were formatted correctly.

In [11]:
# store as float32 lists (ready for Oracle VECTOR column)
dataset_df["embedding"] = [e.astype(np.float32).tolist() for e in embs]

dim = len(dataset_df["embedding"].iloc[0])

# View the first 2 rows of the dataset
dataset_df.head(2)

,id,arxiv_id,title,abstract,text,text_prefixed,embedding
0,arxiv_0,0902.3253,The gravitational wave background from star-ma...,Stars on eccentric orbits around a massive bla...,The gravitational wave background from star-ma...,search_document: The gravitational wave backgr...,"[0.033107940107584, 0.06245188042521477, -0.19..."
1,arxiv_1,0902.0428,Dynamics of planets in retrograde mean motion ...,"In a previous paper (Gayon &amp; Bois 2008a), ...",Dynamics of planets in retrograde mean motion ...,search_document: Dynamics of planets in retrog...,"[0.02952495776116848, 0.07589568942785263, -0...."


----

# Part 3: Database Table Setup and Data Ingestion

This code below drops and recreates a table called research_papers with columns for paper metadata and a VECTOR column that stores embeddings of size `dim` (the embedding dimension you computed earlier), enabling Oracle to perform vector search on those embeddings.

## 3.1 Create Reseach Papers Table

In [12]:
ddl = f"""
BEGIN
    EXECUTE IMMEDIATE 'DROP TABLE research_papers';
EXCEPTION WHEN OTHERS THEN
    IF SQLCODE != -942 THEN RAISE; END IF;
END;
/
CREATE TABLE research_papers (
    arxiv_id VARCHAR2(255) PRIMARY KEY,
    title VARCHAR2(4000),
    abstract VARCHAR2(4000),
    text CLOB,
    embedding VECTOR({dim}, FLOAT32)
)
TABLESPACE USERS
"""

Here, we’re taking the multi-statement SQL stored in `ddl`, splitting it by `/` so each command runs separately, and executing them one by one using the database cursor.  
After all statements finish, we call `conn.commit()` to save the changes — effectively creating the `RESEARCH_PAPERS` table — and then print a confirmation message showing the vector dimension (`dim`) used for the `embedding` column.


In [13]:
with conn.cursor() as cur:
    for stmt in ddl.split("/"):
        if stmt.strip():
            cur.execute(stmt)

conn.commit()
print("✅ Table RESEARCH_PAPERS created with VECTOR dimension:", dim)

✅ Table RESEARCH_PAPERS created with VECTOR dimension: 768


## 3.2 Create Indexes (Vector and Search)

This code in the next cell below creates a **vector index** on the `embedding` column of the `research_papers` table to enable fast similarity search.  
The index, named `RP_VEC_IVF`, uses Oracle’s `VECTOR` indexing with **IVF (Inverted File)** organization, which partitions vectors into clusters for efficient nearest-neighbor lookup.  
It’s configured to use **cosine distance** as the similarity metric and aims for a **target accuracy of 90%**, balancing search speed and precision.  
Finally, `conn.commit()` saves the index creation, and a confirmation message is printed once the index is successfully built.

Note: In Oracle’s vector indexing syntax, **ORGANIZATION NEIGHBOR PARTITIONS** specifies that the index should be built using the IVF (Inverted File) structure — a partition-based nearest-neighbor organization optimized for large-scale vector search.


In [14]:
with conn.cursor() as cur:
    cur.execute("""
        CREATE VECTOR INDEX RP_VEC_IVF
        ON research_papers(embedding)
        ORGANIZATION NEIGHBOR PARTITIONS
        DISTANCE COSINE
        WITH TARGET ACCURACY 90
        TABLESPACE USERS
    """ )

conn.commit()
print("✅ Vector Index RP_VEC_IVF created")

✅ Vector Index RP_VEC_IVF created


> **💡 Knowledge Checkpoint**
>
> IVF (Inverted File Index) is a vector indexing technique that speeds up similarity search by clustering vectors into groups (called partitions or  centroids).
>
> Instead of comparing a query vector to every vector in the dataset, IVF first identifies the most relevant clusters and only searches within those > — drastically reducing computation.
> 
> In Oracle AI Database, the clause ORGANIZATION NEIGHBOR PARTITIONS activates this IVF-style structure, allowing the database to perform approximate nearest-neighbor (ANN) searches efficiently while maintaining high accuracy (controlled by the WITH TARGET ACCURACY parameter).

**Creating an Oracle Text Index for Full-Text Search**

This code below creates a **full-text search index** on the `text` column of the `research_papers` table using Oracle Text.

1. **Drop existing index** — removes `rp_text_idx` if it already exists, ignoring the “index does not exist” error.  
2. **Create new text index** — builds a `CTXSYS.CONTEXT` index, which tokenizes and indexes the text for efficient keyword searches.  
   - `SYNC (ON COMMIT)` keeps the index automatically updated whenever new data is committed.  
3. **Commit and confirm** — saves the changes and prints a success message.

Once created, you can use the `CONTAINS()` operator with `SCORE()` for fast, ranked keyword searches, e.g.:

```sql
SELECT title, SCORE(1) AS relevance
FROM research_papers
WHERE CONTAINS(text, 'transformer architecture', 1) > 0
ORDER BY SCORE(1) DESC;
```

This turns your text column into a search-optimized field, similar to how search engines handle full-text retrieval.


In [15]:
with conn.cursor() as cur:
    # Drop old index if it exists
    try:
        cur.execute("DROP INDEX rp_text_idx")
    except oracledb.DatabaseError as e:
        if "ORA-01418" not in str(e):  # ignore "index does not exist"
            raise

    # Create a TEXT index on the 'text' column only
    cur.execute("""
        CREATE INDEX rp_text_idx
        ON research_papers(text)
        INDEXTYPE IS CTXSYS.CONTEXT
        PARAMETERS ('SYNC (ON COMMIT)')
    """)

conn.commit()
print("✅ Oracle Text index (rp_text_idx) created successfully on TEXT column.")


✅ Oracle Text index (rp_text_idx) created successfully on TEXT column.


In the code below we’re taking all the research paper records (including their embeddings) from the DataFrame and inserting them into the `RESEARCH_PAPERS` table in Oracle.

Let’s break down what’s happening:


1. Convert each embedding for Oracle compatibility
```python
embedding_array = array.array('f', row.get("embedding"))
```
Oracle’s `VECTOR` column expects the data as a compact binary float array, not a Python list.  
So here we convert each embedding into an `array.array('f')` — this ensures the data binds correctly and efficiently when inserting.


2. Prepare all rows for insertion
For every paper, we build a tuple containing its metadata (`arxiv_id`, `title`, `abstract`, `text`) and the formatted `embedding_array`.  
These tuples are collected in a list called `rows`.

3. Insert each row with a progress bar
```python
for row in tqdm(rows):
    cur.execute("""
        INSERT INTO research_papers (arxiv_id, title, abstract, text, embedding)
        VALUES (:1, :2, :3, :4, :5)
    """, row)
```
We loop through each row, inserting it into the table using Oracle’s parameterized query syntax.  
The `tqdm` progress bar gives real-time feedback on the insertion process — helpful when inserting hundreds or thousands of embeddings.


4. Commit everything
Finally, `conn.commit()` saves all the inserted records permanently in the database.  You’ll see a confirmation message once the operation completes successfully.


## 3.3 Ingest Data into Oracle

In [6]:
from tqdm import tqdm
import array

In [ ]:
rows = []
for i, row in dataset_df.iterrows():
    # Convert embedding list to array.array for proper VECTOR binding
    embedding_array = array.array('f', row.get("embedding"))
    
    rows.append((
        row.get("arxiv_id"),
        row.get("title"),
        row.get("abstract"),
        row.get("text"),
        embedding_array
    ))

print(f"Preparing to insert {len(rows)} rows into RESEARCH_PAPERS...")

with conn.cursor() as cur:
    for row in tqdm(rows):
        cur.execute(
            """
            INSERT INTO research_papers (arxiv_id, title, abstract, text, embedding)
            VALUES (:1, :2, :3, :4, :5)
            """, 
            row
        )
        
conn.commit()
print("✅ Data inserted successfully")

Preparing to insert 1000 rows into RESEARCH_PAPERS...


100%|██████████| 1000/1000 [00:03<00:00, 286.80it/s]


✅ Data inserted successfully


In [7]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM RESEARCH_PAPERS")
    print("Row count:", cur.fetchone()[0])

    cur.execute("""
        SELECT arxiv_id, title, abstract, text FROM RESEARCH_PAPERS
        FETCH FIRST 3 ROWS ONLY
    """)
    for row in cur.fetchall():
        print(row)

Row count: 1000
('0902.0428', 'Dynamics of planets in retrograde mean motion resonance', 'In a previous paper (Gayon &amp; Bois 2008a), we have shown the general efficiency of retrograde resonances for stabilizing compact planetary systems. Such retrograde resonances can be found when two-planets of a three-body planetary system are both in mean motion resonance and revolve in opposite directions. For a particular two-planet system, we have also obtained a new orbital fit involving such a counter-revolving configuration and consistent with the observational data. <br>In the present paper, we analytically investigate the three-body problem in this particular case of retrograde resonances. We therefore define a new set of canonical variables allowing to express correctly the resonance angles and obtain the Hamiltonian of a system harboring planets revolving in opposite directions. The acquiring of an analytical &#34;rail&#34; may notably contribute to a deeper understanding of our numeri

# Part 4. Retrieval Mechanisms

In [8]:
SEARCH_TEXT_KEYWORDS = "optimization"

## 4.1 Text Based Retrieval

The code below performs a full-text search over the text column of the research_papers table, using the Oracle Text index we just created earlier

- CONTAINS(text, :keyword, 1) uses the Oracle Text index on the text column to find documents containing the given keyword or phrase.
- SCORE(1) assigns a relevance score based on how well each document matches the search term.
- SUBSTR(text, 1, 200) returns the first 200 characters as a short preview snippet.
- FETCH FIRST 10 ROWS ONLY limits the results to the top 10 most relevant matches.

In [9]:
def keyword_search_research_papers(conn, keyword: str):
    """
    Perform a full-text keyword search on the 'text' column 
    using the Oracle Text index (rp_text_idx).

    Args:
        conn: Oracle database connection object.
        keyword (str): Keyword or phrase to search for.

    Returns:
        tuple: (rows, columns)
    """
    query = """
        SELECT 
            arxiv_id, 
            title, 
            SUBSTR(text, 1, 200) AS text_snippet,
            SCORE(1) AS relevance_score
        FROM research_papers
        WHERE CONTAINS(text, :keyword, 1) > 0
        ORDER BY SCORE(1) DESC
        FETCH FIRST 10 ROWS ONLY
    """

    with conn.cursor() as cur:
        cur.execute(query, keyword=keyword)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    return rows, columns


In [7]:
rows, columns = keyword_search_research_papers(conn, SEARCH_TEXT_KEYWORDS)

results_df = pd.DataFrame(rows, columns=columns)

print(f"🔍 Keyword Search: '{SEARCH_TEXT_KEYWORDS}'")
print(f"📊 Results: {len(results_df)}\n")

results_df


🔍 Keyword Search: 'optimization'
📊 Results: 3



,ARXIV_ID,TITLE,TEXT_SNIPPET,RELEVANCE_SCORE
0,0912.1394,Are Near Earth Objects the Key to Optimization...,Are Near Earth Objects the Key to Optimization...,42
1,0901.0846,A Metric and Optimisation Scheme for Microlens...,A Metric and Optimisation Scheme for Microlens...,11
2,0904.4106,Is the HR 8799 extrasolar system destined for ...,Is the HR 8799 extrasolar system destined for ...,11


As shown above, the returned rows represent documents whose indexed `text` content matched the full-text search query **"Optimization"**, as determined by the Oracle Text `CONTAINS()` operator using the `rp_text_idx` index.


## 4.2 Vector Based Retrieval

In [10]:
SEARCH_QUERY = "Get me papers related to planetary exploration"

**Vector Similarity Search in Oracle**

This function below `vector_search_research_papers` performs a **semantic vector search** on the `research_papers` table, retrieving papers most similar to a given query using **Oracle’s native VECTOR search** feature.


Step-by-step overview

1. Encode the search query
```python
query_embedding = embedding_model.encode(
    [f"search_query: {search_query}"],
    convert_to_numpy=True,
    normalize_embeddings=True
)[0].astype(np.float32).tolist()
```
- The query is embedded using the same model that generated the document embeddings.  
- The prefix `"search_query:"` ensures embeddings align with `"search_document:"` vectors.  
- Normalized and cast to `float32` for Oracle’s `VECTOR` type.


2. Prepare for database binding
```python
query_embedding_array = array.array('f', query_embedding)
```
- Converts the embedding list into a binary float array (`array('f')`), required for Oracle’s vector binding.


3. Perform vector search
```sql
SELECT arxiv_id, title, abstract,
       SUBSTR(text, 1, 200) AS text_snippet,
        ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity_score
FROM research_papers
ORDER BY similarity_score
FETCH APPROX FIRST {top_k} ROWS ONLY WITH TARGET ACCURACY 90
```
- Computes **cosine distance** between the query vector and stored embeddings.  
- Orders results by similarity, returning the closest matches.  
- Uses **Approximate Nearest Neighbor (ANN)** search for fast retrieval at 90% target accuracy.

In [11]:
def vector_search_research_papers(conn, embedding_model, search_query: str, top_k: int = 5):
    """
    Perform a vector similarity search on the research_papers table using a query embedding.
    Returns cosine similarity scores (higher = more similar).
    """

    # 1️⃣ Encode the query into a vector
    query_embedding = embedding_model.encode(
        [f"search_query: {search_query}"], 
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype(np.float32).tolist()

    # 2️⃣ Prepare the vector for Oracle binding
    query_embedding_array = array.array('f', query_embedding)

    # 3️⃣ Run a vector similarity search using cosine similarity
    query = f"""
        SELECT 
            arxiv_id, 
            title, 
            abstract, 
            SUBSTR(text, 1, 200) AS text_snippet,
            ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity_score
        FROM research_papers
        ORDER BY similarity_score DESC
        FETCH APPROX FIRST {top_k} ROWS ONLY WITH TARGET ACCURACY 90
    """

    # 4️⃣ Execute and return results
    with conn.cursor() as cur:
        cur.execute(query, q=query_embedding_array)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    return rows, columns


In [14]:
rows, columns = vector_search_research_papers(conn, embedding_model, SEARCH_TEXT_KEYWORDS, top_k=5)

results_df = pd.DataFrame(rows, columns=columns)

print(f"🔍 Vector Search: '{SEARCH_TEXT_KEYWORDS}'")
print(f"📊 Results: {len(results_df)}\n")

results_df


🔍 Vector Search: 'optimization'
📊 Results: 5



,ARXIV_ID,TITLE,ABSTRACT,TEXT_SNIPPET,SIMILARITY_SCORE
0,0912.1394,Are Near Earth Objects the Key to Optimization...,This note suggests that near earth objects and...,Are Near Earth Objects the Key to Optimization...,0.7392
1,0901.0846,A Metric and Optimisation Scheme for Microlens...,OGLE III and MOA II are discovering 600-1000 G...,A Metric and Optimisation Scheme for Microlens...,0.6409
2,0911.2703,An Efficient Method for Modeling High Magnific...,I present a previously unpublished method for ...,An Efficient Method for Modeling High Magnific...,0.6136
3,1001.0437,Kepler Science Operations,Kepler&#39;s primary mission is a search for e...,Kepler Science Operations — Kepler&#39;s prima...,0.6019
4,0903.5139,Science-Operational Metrics and Issues for the...,A movement is underway to test the uniqueness ...,Science-Operational Metrics and Issues for the...,0.6009


In this step, we used Oracle Database’s native vector search capabilities to perform a semantic similarity search over the research_papers dataset.

The text query — “Get me papers related to planetary exploration” — was first transformed into a high-dimensional embedding vector using the same model that generated our document embeddings (nomic-embed-text-v1.5).
This query vector captures the semantic meaning of the phrase rather than just the exact words.

Using Oracle’s VECTOR_DISTANCE(..., COSINE) function (converted to 1 - distance), we then compared this query vector against the stored embeddings for each paper in the database.
The result is a ranked list of research papers ordered by cosine similarity, where higher scores indicate stronger semantic alignment with the search query.

## 4.3 Hybrid Retrieval (Vector + Text Combined)

#### 4.3.1 Pre Filtering

In [12]:
import array
import numpy as np

def hybrid_search_research_papers_pre_filter(
    conn,
    embedding_model,
    search_phrase: str,
    top_k: int = 10,
    show_explain: bool = False
):
    """
    Perform a hybrid search using Oracle Text + Vector Search.
    Combines lexical filtering (CONTAINS) with semantic re-ranking via cosine similarity.

    Args:
        conn: Oracle database connection object.
        embedding_model: Model with `.encode()` method (e.g., SentenceTransformer).
        search_phrase (str): User search phrase used for both text filtering and embedding.
        top_k (int): Number of results to return (default = 10).
        show_explain (bool): If True, prints the execution plan.

    Returns:
        tuple: (rows, columns, exec_plan_text or None)
    """

    # --- Step 1: Encode search phrase into normalized vector ---
    query_embedding = embedding_model.encode(
        [f"search_query: {search_phrase}"],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype(np.float32).tolist()
    query_embedding_array = array.array('f', query_embedding)

    with conn.cursor() as cur:
        # Enable runtime stats if needed
        if show_explain:
            cur.execute("ALTER SESSION SET statistics_level = ALL")

        # --- Step 2: Hybrid query (Oracle Text + Vector) ---
        sql = f"""
            SELECT {"/*+ GATHER_PLAN_STATISTICS */" if show_explain else ""}
                arxiv_id,
                title,
                abstract,
                SUBSTR(text, 1, 200) AS text_snippet,
                ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity_score
            FROM research_papers
            WHERE CONTAINS(text, :kw, 1) > 0
            ORDER BY similarity_score DESC
            FETCH APPROX FIRST {top_k} ROWS ONLY WITH TARGET ACCURACY 90
        """

        cur.execute(sql, q=query_embedding_array, kw=search_phrase)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    # --- Step 3: Execution plan (optional) ---
    exec_plan_text = None
    if show_explain:
        with conn.cursor() as cur_plan:
            cur_plan.execute("""
                SELECT plan_table_output
                FROM TABLE(DBMS_XPLAN.DISPLAY_CURSOR(NULL, NULL, 'ALLSTATS LAST +PREDICATE'))
            """)
            exec_plan_text = "\n".join(r[0] for r in cur_plan.fetchall())

        print("\n====== Execution Plan (DBMS_XPLAN.DISPLAY_CURSOR) ======")
        print(exec_plan_text)
        print("========================================================\n")

    return rows, columns, exec_plan_text


In [17]:
rows, columns, exec_plan = hybrid_search_research_papers_pre_filter(
    conn,
    embedding_model,
    search_phrase=SEARCH_TEXT_KEYWORDS,
    top_k=10,
    show_explain=False
)

pre_filter_results_df = pd.DataFrame(rows, columns=columns)

print(f"🔍 Hybrid Search: '{SEARCH_TEXT_KEYWORDS}'")
print(f"📊 Found {len(pre_filter_results_df)} results\n")

pre_filter_results_df


🔍 Hybrid Search: 'optimization'
📊 Found 3 results



,ARXIV_ID,TITLE,ABSTRACT,TEXT_SNIPPET,SIMILARITY_SCORE
0,0912.1394,Are Near Earth Objects the Key to Optimization...,This note suggests that near earth objects and...,Are Near Earth Objects the Key to Optimization...,0.7392
1,0901.0846,A Metric and Optimisation Scheme for Microlens...,OGLE III and MOA II are discovering 600-1000 G...,A Metric and Optimisation Scheme for Microlens...,0.6409
2,0904.4106,Is the HR 8799 extrasolar system destined for ...,The recent discovery of a three-planet extraso...,Is the HR 8799 extrasolar system destined for ...,0.5576


#### 4.3.2 Post Filtering

In [13]:
import array
import numpy as np

def hybrid_search_research_papers_postfilter(
    conn,
    embedding_model,
    search_phrase: str,
    top_k: int = 10,
    candidate_k: int = 200,
    show_explain: bool = False
):
    """
    Perform a hybrid search using Vector Search first, then Oracle Text filtering.
    Returns top results ranked by semantic similarity but filtered by lexical match.

    Args:
        conn: Oracle database connection object.
        embedding_model: Model with `.encode()` method (e.g., SentenceTransformer).
        search_phrase (str): Search phrase used for both embedding and text filtering.
        top_k (int): Number of top results to return (default = 10).
        candidate_k (int): Number of initial vector candidates (default = 200).
        show_explain (bool): If True, prints the execution plan.

    Returns:
        tuple: (rows, columns, exec_plan_text or None)
    """

    # --- Step 1: Encode search phrase into a normalized query vector ---
    query_embedding = embedding_model.encode(
        [f"search_query: {search_phrase}"],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype(np.float32).tolist()
    query_embedding_array = array.array('f', query_embedding)

    with conn.cursor() as cur:
        # Enable runtime statistics if requested
        if show_explain:
            cur.execute("ALTER SESSION SET statistics_level = ALL")

        # --- Step 2: Hybrid query (Vector first → Text filter) ---
        sql = f"""
            WITH vec_candidates AS (
                SELECT
                    arxiv_id,
                    title,
                    abstract,
                    text,
                    1 - VECTOR_DISTANCE(embedding, :q, COSINE) AS similarity_score
                FROM research_papers
                ORDER BY similarity_score DESC
                FETCH APPROX FIRST {candidate_k} ROWS ONLY WITH TARGET ACCURACY 90
            )
            SELECT {"/*+ GATHER_PLAN_STATISTICS */" if show_explain else ""}
                arxiv_id,
                title,
                SUBSTR(text, 1, 200) AS text_snippet,
                ROUND(similarity_score, 4) AS similarity_score
            FROM vec_candidates
            WHERE CONTAINS(text, :kw, 1) > 0
            ORDER BY similarity_score DESC
            FETCH FIRST {top_k} ROWS ONLY
        """

        cur.execute(sql, q=query_embedding_array, kw=search_phrase)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    # --- Step 3: Fetch and display execution plan (optional) ---
    exec_plan_text = None
    if show_explain:
        with conn.cursor() as cur_plan:
            cur_plan.execute("""
                SELECT plan_table_output
                FROM TABLE(DBMS_XPLAN.DISPLAY_CURSOR(NULL, NULL, 'ALLSTATS LAST +PREDICATE'))
            """)
            exec_plan_text = "\n".join(r[0] for r in cur_plan.fetchall())

        print("\n====== Execution Plan (DBMS_XPLAN.DISPLAY_CURSOR) ======")
        print(exec_plan_text)
        print("========================================================\n")

    return rows, columns, exec_plan_text


In [19]:
rows, columns, exec_plan = hybrid_search_research_papers_postfilter(
    conn,
    embedding_model,
    search_phrase=SEARCH_TEXT_KEYWORDS,
    top_k=10,
    show_explain=False
)

post_filter_results_df = pd.DataFrame(rows, columns=columns)

print(f"🔍 Hybrid Search: '{SEARCH_TEXT_KEYWORDS}'")
print(f"📊 Found {len(post_filter_results_df)} results\n")

post_filter_results_df


🔍 Hybrid Search: 'optimization'
📊 Found 3 results



,ARXIV_ID,TITLE,TEXT_SNIPPET,SIMILARITY_SCORE
0,0912.1394,Are Near Earth Objects the Key to Optimization...,Are Near Earth Objects the Key to Optimization...,0.7392
1,0901.0846,A Metric and Optimisation Scheme for Microlens...,A Metric and Optimisation Scheme for Microlens...,0.6409
2,0904.4106,Is the HR 8799 extrasolar system destined for ...,Is the HR 8799 extrasolar system destined for ...,0.5576


Observe pre/post filtering technques in a table side by side 

In [20]:
# Show the results side by side
pd.concat([pre_filter_results_df, post_filter_results_df], axis=1)


,ARXIV_ID,TITLE,ABSTRACT,TEXT_SNIPPET,SIMILARITY_SCORE,ARXIV_ID,TITLE,TEXT_SNIPPET,SIMILARITY_SCORE
0,0912.1394,Are Near Earth Objects the Key to Optimization...,This note suggests that near earth objects and...,Are Near Earth Objects the Key to Optimization...,0.7392,0912.1394,Are Near Earth Objects the Key to Optimization...,Are Near Earth Objects the Key to Optimization...,0.7392
1,0901.0846,A Metric and Optimisation Scheme for Microlens...,OGLE III and MOA II are discovering 600-1000 G...,A Metric and Optimisation Scheme for Microlens...,0.6409,0901.0846,A Metric and Optimisation Scheme for Microlens...,A Metric and Optimisation Scheme for Microlens...,0.6409
2,0904.4106,Is the HR 8799 extrasolar system destined for ...,The recent discovery of a three-planet extraso...,Is the HR 8799 extrasolar system destined for ...,0.5576,0904.4106,Is the HR 8799 extrasolar system destined for ...,Is the HR 8799 extrasolar system destined for ...,0.5576


| Approach                          | Strength                            | Best For                                |
| --------------------------------- | ----------------------------------- | --------------------------------------- |
| **Pre-filter** (`CONTAINS` first) | Fast, keyword-strict                | Narrow keyword search                   |
| **Post-filter** (this one)        | Semantically rich but still precise | Broader exploratory or research queries |


#### 4.3.3 Reciprocial Rank Fusion

In [14]:
import array
import numpy as np
import oracledb

def hybrid_rrf_search(
    conn,
    embedding_model,
    search_phrase: str,
    top_k: int = 10,
    per_list: int = 120,     # candidates from each list before fusion (>= 10x top_k is a good rule)
    k: int = 60,             # RRF smoothing constant (60 is standard)
    phrase_safe: bool = True,
    show_explain: bool = False
):
    """
    Local-friendly RRF fusion of Vector + Oracle Text results on research_papers(text, embedding).

    Prereqs (local/Docker Free OK):
      - VECTOR column/index on research_papers(embedding)  -- IVF/HNSW
      - Oracle Text index on research_papers(text)         -- e.g. CREATE SEARCH INDEX rp_text_idx ON research_papers(text);

    RRF = 1/(k + r_vec) + 1/(k + r_txt), where r_vec and r_txt are ranks (1 = best).
    """

    # 1) Encode query for vector modality (align with your doc prefixing scheme)
    qv = embedding_model.encode(
        [f"search_query: {search_phrase}"],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype(np.float32).tolist()
    qv = array.array('f', qv)

    # 2) Phrase-safe text query for Oracle Text (optional)
    kw = f"\"{search_phrase}\"" if (phrase_safe and " " in search_phrase.strip()) else search_phrase

    with conn.cursor() as cur:
        if show_explain:
            cur.execute("ALTER SESSION SET statistics_level = ALL")

        sql = f"""
            WITH
            /* Vector top-N with ranks (higher similarity first) */
            vec AS (
              SELECT
                arxiv_id,
                title,
                SUBSTR(text, 1, 200) AS text_snippet,
                1 - VECTOR_DISTANCE(embedding, :q, COSINE) AS sim_vec,
                ROW_NUMBER() OVER (ORDER BY 1 - VECTOR_DISTANCE(embedding, :q, COSINE) DESC) AS r_vec
              FROM research_papers
              FETCH APPROX FIRST {per_list} ROWS ONLY WITH TARGET ACCURACY 90
            ),
            /* Oracle Text top-N with ranks (higher SCORE(1) first) */
            txt AS (
              SELECT
                arxiv_id,
                title,
                SUBSTR(text, 1, 200) AS text_snippet,
                SCORE(1) AS score_txt,
                ROW_NUMBER() OVER (ORDER BY SCORE(1) DESC) AS r_txt
              FROM research_papers
              WHERE CONTAINS(text, :kw, 1) > 0
              FETCH FIRST {per_list} ROWS ONLY
            ),
            /* Fuse by arxiv_id; keep docs present in either list */
            fused AS (
              SELECT
                COALESCE(v.arxiv_id, t.arxiv_id)           AS arxiv_id,
                COALESCE(v.title,     t.title)             AS title,
                COALESCE(v.text_snippet, t.text_snippet)   AS text_snippet,
                NVL(v.r_vec,  999999) AS r_vec,
                NVL(t.r_txt,  999999) AS r_txt,
                NVL(v.sim_vec, 0)     AS sim_vec,
                NVL(t.score_txt, 0)   AS score_txt
              FROM vec v
              FULL OUTER JOIN txt t
                ON t.arxiv_id = v.arxiv_id
            )
            SELECT {"/*+ GATHER_PLAN_STATISTICS */" if show_explain else ""}
              arxiv_id,
              title,
              text_snippet,
              ROUND( (1.0/(:k + r_vec)) + (1.0/(:k + r_txt)), 6 ) AS rrf_score,
              r_vec,
              r_txt,
              ROUND(sim_vec, 4)  AS sim_vec,
              ROUND(score_txt,4) AS score_txt
            FROM fused
            ORDER BY rrf_score DESC, title
            FETCH FIRST {top_k} ROWS ONLY
        """

        cur.execute(sql, q=qv, kw=kw, k=k)
        rows = cur.fetchall()
        columns = [d[0] for d in cur.description]

    exec_plan_text = None
    if show_explain:
        with conn.cursor() as cur_plan:
            cur_plan.execute("""
                SELECT plan_table_output
                FROM TABLE(DBMS_XPLAN.DISPLAY_CURSOR(NULL, NULL, 'ALLSTATS LAST +PREDICATE'))
            """)
            exec_plan_text = "\n".join(r[0] for r in cur_plan.fetchall())
            print("\n====== Execution Plan (DBMS_XPLAN.DISPLAY_CURSOR) ======")
            print(exec_plan_text)
            print("========================================================\n")

    return rows, columns, exec_plan_text


In [47]:
rows, columns, exec_plan = hybrid_rrf_search(
    conn,
    embedding_model,
    search_phrase=SEARCH_TEXT_KEYWORDS,
    top_k=3,
    show_explain=False
)

rrf_results_df = pd.DataFrame(rows, columns=columns)

print(f"🔍 Hybrid Search: '{SEARCH_TEXT_KEYWORDS}'")
print(f"📊 Found {len(rrf_results_df)} results\n")

rrf_results_df


🔍 Hybrid Search: 'optimization'
📊 Found 3 results



,ARXIV_ID,TITLE,TEXT_SNIPPET,RRF_SCORE,R_VEC,R_TXT,SIM_VEC,SCORE_TXT
0,0912.1394,Are Near Earth Objects the Key to Optimization...,Are Near Earth Objects the Key to Optimization...,0.032787,1,1,0.7392,42
1,0901.0846,A Metric and Optimisation Scheme for Microlens...,A Metric and Optimisation Scheme for Microlens...,0.032002,2,3,0.6409,11
2,0904.4106,Is the HR 8799 extrasolar system destined for ...,Is the HR 8799 extrasolar system destined for ...,0.021811,116,2,0.5576,11


# Part 5: Building a RAG pipeline



## Azure OpenAI equivalent for Part 5

The following companion cells reproduce the same flow using Azure OpenAI.
Run the Azure cell immediately after each original Section 5 cell.

In [15]:
import getpass
import os

# Function to securely get and set environment variables
def set_env_securely(var_name, prompt):
    value = getpass.getpass(prompt)
    os.environ[var_name] = value

In [16]:
# Azure OpenAI environment setup helpers
import getpass
import os

# Function to securely get and set environment variables
def set_env_securely_azure(var_name, prompt):
    value = getpass.getpass(prompt)
    os.environ[var_name] = value

In [ ]:
set_env_securely("OPENAI_API_KEY", "Enter your OPEN API Key: ")

In [17]:
# https://azure-agent-ai-foundry-resource.openai.azure.com/
# gpt-4o
# gpt-4.1
# gpt-5

set_env_securely_azure("AZURE_OPENAI_ENDPOINT", "Enter your Azure OpenAI endpoint (e.g. https://<resource>.openai.azure.com): ")
set_env_securely_azure("AZURE_OPENAI_DEPLOYMENT", "Enter your Azure OpenAI deployment name (e.g. gpt-4o): ")
print("RBAC auth enabled: ensure you are signed in (for example, via 'az login') and have Azure OpenAI permissions.")

RBAC auth enabled: ensure you are signed in (for example, via 'az login') and have Azure OpenAI permissions.


In [ ]:
# Import the OpenAI Python client library
from openai import OpenAI
import os

# Initialize the OpenAI client (API key read from env var OPENAI_API_KEY)
openai_client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
)

# Create a Response API request (using the Responses API)

# Use the Responses API
response = openai_client.responses.create(
    model="gpt-4o",              # specify model
    input="Hello! I’m a user!",  # user message as a text input
    instructions="You are a research paper assistant.",  # assistant role/instruction
)

In [18]:
# Import the OpenAI Python client library for Azure OpenAI
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import os

# Initialize Azure OpenAI client with RBAC (Microsoft Entra ID)
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)
azure_openai_client = OpenAI(
    api_key=token_provider,
    base_url=f"{os.environ.get('AZURE_OPENAI_ENDPOINT', '').rstrip('/')}/openai/v1/",
)

# Create a Responses API request against your Azure deployment
azure_response = azure_openai_client.responses.create(
    model=os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o"),
    input="Hello! I’m a user!",
    instructions="You are a research paper assistant.",
)

In [ ]:
# Print the output text
print(response.output_text)


In [19]:
# Print the Azure OpenAI output text
print(azure_response.output_text)


Hello! How can I assist you today? Are you working on a research paper, exploring ideas, or looking for guidance on something specific? Let me know!


In [43]:
def research_paper_assistant_rag_pipeline(
    conn,
    embedding_model,
    user_query: str,
    top_k: int = 10,
    retrieval_mode: str = "hybrid",
    show_explain: bool = False
):
    """
    Research Paper Assistant — Retrieval-Augmented Generation (RAG) pipeline
    built on SQL-based retrieval functions and powered by the OpenAI Responses API.

    Retrieval techniques available:
        - 'keyword'  → uses keyword_search_research_papers()
        - 'vector'   → uses vector_search_research_papers()
        - 'hybrid'   → uses hybrid_search_research_papers() [default]

    Args:
        conn: Oracle database connection.
        embedding_model: Embedding model (e.g., SentenceTransformer, Voyage).
        user_query (str): Research question from the user.
        top_k (int): Number of top documents to retrieve.
        retrieval_mode (str): Retrieval method ('keyword', 'vector', 'hybrid').
        show_explain (bool): Whether to show the SQL execution plan.

    Returns:
        str: LLM-generated research synthesis with citations.
    """

    # ----------------------------------------------------------------------
    # 1. Retrieve relevant research papers using the selected retrieval mode
    # ----------------------------------------------------------------------
    if retrieval_mode == "keyword":
        rows, columns = keyword_search_research_papers(conn, user_query)
        exec_plan_text = None

    elif retrieval_mode == "vector":
        rows, columns = vector_search_research_papers(conn, embedding_model, user_query, top_k)
        exec_plan_text = None

    else:  # default: hybrid retrieval
        rows, columns, exec_plan_text = hybrid_search_research_papers_pre_filter(
            conn=conn,
            embedding_model=embedding_model,
            search_phrase=user_query,
            top_k=top_k,
            show_explain=show_explain
        )

    retrieved_count = len(rows) if rows else 0
    print(f"📊 Retrieved {retrieved_count} papers using {retrieval_mode.upper()} retrieval.")


    # ----------------------------------------------------------------------
    # 2. Convert retrieved rows to formatted LLM context
    # ----------------------------------------------------------------------
    formatted_context = ""
    if retrieved_count > 0:
        formatted_context += f"\n\n📚 {retrieved_count} relevant research papers retrieved:\n\n"
        for i, row in enumerate(rows):
            row_data = dict(zip(columns, row))
            title = row_data.get("TITLE", "Untitled Paper")
            abstract = row_data.get("ABSTRACT", "No abstract available.")
            snippet = row_data.get("TEXT_SNIPPET", "")
            score = (
                row_data.get("SIMILARITY_SCORE")
                or row_data.get("TEXT_RELEVANCE_SCORE")
                or "N/A"
            )
            formatted_context += (
                f"[{i+1}] **{title}**\n"
                f"Abstract: {abstract}\n"
                f"Snippet: {snippet}\n"
                f"Relevance Score: {score}\n\n"
            )
    else:
        formatted_context = "\n\n⚠️ No relevant papers were retrieved from the database.\n"

    # ----------------------------------------------------------------------
    # 3. Construct the prompt for the Responses API
    # ----------------------------------------------------------------------
    prompt = f"""
            You are a **Research Paper Assistant** that synthesizes academic literature to help answer user questions.

            User Query: {user_query}

            Number of retrieved papers: {retrieved_count}
            {formatted_context}

            Please:
            - Summarize the findings most relevant to the query.
            - Use citation numbers [X] to support claims.
            - Highlight consensus, innovation, or research gaps.
            - If there is insufficient context, clearly say so.
            """

    # ----------------------------------------------------------------------
    # 4. Call the OpenAI Responses API
    # ----------------------------------------------------------------------
    response = openai_client.responses.create(
        model="gpt-4o",
        input=prompt,
        instructions="You are a scientific research assistant. Use only the provided context to answer. Always cite papers [1], [2], etc.",
        temperature=0.3,
    )

    # ----------------------------------------------------------------------
    # 5. Optionally print SQL execution plan (if hybrid)
    # ----------------------------------------------------------------------
    if show_explain and exec_plan_text:
        print("\n====== SQL Execution Plan ======")
        print(exec_plan_text)
        print("================================\n")

    # ----------------------------------------------------------------------
    # 6. Return the LLM’s output text
    # ----------------------------------------------------------------------
    return response.output_text


In [20]:
def research_paper_assistant_rag_pipeline_azure(
    conn,
    embedding_model,
    user_query: str,
    top_k: int = 10,
    retrieval_mode: str = "hybrid",
    show_explain: bool = False
):
    """
    Research Paper Assistant — Retrieval-Augmented Generation (RAG) pipeline
    built on SQL-based retrieval functions and powered by Azure OpenAI Responses API.
    """

    # 1. Retrieve relevant research papers using the selected retrieval mode
    if retrieval_mode == "keyword":
        rows, columns = keyword_search_research_papers(conn, user_query)
        exec_plan_text = None

    elif retrieval_mode == "vector":
        rows, columns = vector_search_research_papers(conn, embedding_model, user_query, top_k)
        exec_plan_text = None

    else:  # default: hybrid retrieval
        rows, columns, exec_plan_text = hybrid_search_research_papers_pre_filter(
            conn=conn,
            embedding_model=embedding_model,
            search_phrase=user_query,
            top_k=top_k,
            show_explain=show_explain
        )

    retrieved_count = len(rows) if rows else 0
    print(f"📊 Retrieved {retrieved_count} papers using {retrieval_mode.upper()} retrieval.")

    # 2. Convert retrieved rows to formatted LLM context
    formatted_context = ""
    if retrieved_count > 0:
        formatted_context += f"\n\n📚 {retrieved_count} relevant research papers retrieved:\n\n"
        for i, row in enumerate(rows):
            row_data = dict(zip(columns, row))
            title = row_data.get("TITLE", "Untitled Paper")
            abstract = row_data.get("ABSTRACT", "No abstract available.")
            snippet = row_data.get("TEXT_SNIPPET", "")
            score = (
                row_data.get("SIMILARITY_SCORE")
                or row_data.get("TEXT_RELEVANCE_SCORE")
                or "N/A"
            )
            formatted_context += (
                f"[{i+1}] **{title}**\n"
                f"Abstract: {abstract}\n"
                f"Snippet: {snippet}\n"
                f"Relevance Score: {score}\n\n"
            )
    else:
        formatted_context = "\n\n⚠️ No relevant papers were retrieved from the database.\n"

    print("Starting RAG step")

    # 3. Construct the prompt for the Responses API
    prompt = f"""
            You are a **Research Paper Assistant** that synthesizes academic literature to help answer user questions.


            User Query: {user_query}

            Number of retrieved papers: {retrieved_count}
            {formatted_context}

            Please:
            - Summarize the findings most relevant to the query.
            - Use citation numbers [X] to support claims.
            - Highlight consensus, innovation, or research gaps.
            - If there is insufficient context, clearly say so.
            """

    # 4. Call Azure OpenAI Responses API
    response = azure_openai_client.responses.create(
        model=os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o"),
        input=prompt,
        instructions="You are a scientific research assistant. Use only the provided context to answer. Always cite papers [1], [2], etc.",
        temperature=0.3,
    )

    # 5. Optionally print SQL execution plan (if hybrid)
    if show_explain and exec_plan_text:
        print("\n====== SQL Execution Plan ======")
        print(exec_plan_text)
        print("================================\n")

    # 6. Return the LLM output text
    return response.output_text


In [ ]:
summary = research_paper_assistant_rag_pipeline(
    conn=conn,
    embedding_model=embedding_model,
    user_query=SEARCH_TEXT_KEYWORDS,
    top_k=5,
    retrieval_mode="hybrid",  # options: 'keyword', 'vector', 'hybrid'
    show_explain=False
)

print(summary)

In [21]:
summary_azure = research_paper_assistant_rag_pipeline_azure(
    conn=conn,
    embedding_model=embedding_model,
    user_query=SEARCH_TEXT_KEYWORDS,
    top_k=5,
    retrieval_mode="hybrid",  # options: 'keyword', 'vector', 'hybrid'
    show_explain=False
)

print(summary_azure)

📊 Retrieved 3 papers using HYBRID retrieval.
Starting RAG step
### Summary of Findings Relevant to "Optimization"

1. **Central Force Optimization and Near Earth Objects (NEOs)**:  
   Paper [1] explores the connection between optimization theory and Near Earth Objects (NEOs) through Central Force Optimization (CFO). CFO is a deterministic optimization method inspired by Newton's laws, which uses metaphorical "gravity" to locate global maxima of functions. The study highlights similarities between CFO's oscillatory behavior (Davg curves) and NEOs' Delta-V curves, suggesting that NEO theory may address challenges in optimization, such as local trapping and convergence proofs. This innovative analogy could provide new insights into deterministic optimization methods [1].

2. **Microlens Planet Searches Optimization**:  
   Paper [2] proposes an optimization scheme for prioritizing microlens planet searches. The method uses an automatic algorithm to maximize planet detection capabilities 

# Part 6: AI Agents with OpenAI and Oracle AI Database


In [1]:
!pip install -Uq openai-agents

In [22]:
# Azure companion: install Azure identity support used by Azure OpenAI auth options
!pip install -Uq azure-identity

In [27]:
OPENAI_MODEL = "gpt-4"

In [23]:
# Azure companion: use deployment name (not base model name)
AZURE_OPENAI_MODEL = "gpt-4.1"  # replace with your Azure OpenAI deployment name

In [31]:
from agents import Agent, Runner

In [ ]:
research_paper_assistant = Agent(
    name="Research Paper Assistant",
    model=OPENAI_MODEL,
    instructions="""
      You are a Research Paper Assistant focused on helping users explore, analyze, and summarize
      academic research.

      Maintain a professional, concise, and scholarly tone appropriate for research discussions.
    """,
)


In [29]:
# Azure companion: configure openai-agents to use Azure OpenAI via DefaultAzureCredential
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AsyncAzureOpenAI
from agents import Agent, set_default_openai_client, set_tracing_disabled

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default",
)

# Normalize endpoint in case env var includes /openai or deployment path.
raw_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
if "/openai" in raw_endpoint.lower():
    raw_endpoint = raw_endpoint[: raw_endpoint.lower().index("/openai")]

AZURE_OPENAI_MODEL = os.environ["AZURE_OPENAI_DEPLOYMENT"]
# Runner uses Responses API; 2024-10-21 commonly fails on /responses in Azure.
env_api_version = os.environ.get("AZURE_OPENAI_API_VERSION")
if not env_api_version or env_api_version == "2024-10-21":
    AZURE_OPENAI_API_VERSION = "2025-03-01-preview"
else:
    AZURE_OPENAI_API_VERSION = env_api_version

azure_agents_client = AsyncAzureOpenAI(
    azure_endpoint=raw_endpoint,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_ad_token_provider=token_provider,
    # Compatibility for openai<2.x credential gate when using only AAD token provider.
    _enforce_credentials=False,
)

# Guard: fail fast if this ever gets replaced with a sync client
if not isinstance(azure_agents_client, AsyncAzureOpenAI):
    raise TypeError(
        "Expected AsyncAzureOpenAI for agent runs. Restart kernel and rerun this cell before Azure runs."
    )

print(type(azure_agents_client))
print(f"Azure endpoint: {raw_endpoint}")
print(f"Azure deployment: {AZURE_OPENAI_MODEL}")
print(f"Azure API version: {AZURE_OPENAI_API_VERSION}")

# Route Agent/Runner calls to Azure OpenAI for the Azure companion cells
set_default_openai_client(azure_agents_client)
set_tracing_disabled(disabled=True)

research_paper_assistant_azure = Agent(
    name="Research Paper Assistant (Azure)",
    model=AZURE_OPENAI_MODEL,
    instructions="""
      You are a Research Paper Assistant focused on helping users explore, analyze, and summarize
      academic research.

      Maintain a professional, concise, and scholarly tone appropriate for research discussions.
    """,
)

<class 'openai.lib.azure.AsyncAzureOpenAI'>
Azure endpoint: https://azure-agent-ai-foundry-resource.openai.azure.com
Azure deployment: gpt-4o
Azure API version: 2025-03-01-preview


In [51]:
# Optional helper: switch the agents runtime back to standard OpenAI
from openai import AsyncOpenAI

def switch_agents_runtime_to_openai():
    if not os.environ.get("OPENAI_API_KEY"):
        raise ValueError("OPENAI_API_KEY is required to switch back to standard OpenAI.")
    set_default_openai_client(AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"]))
    set_tracing_disabled(disabled=True)
    print("Switched agents runtime back to standard OpenAI client.")

In [ ]:
run_result = await Runner.run(
    starting_agent=research_paper_assistant,
    input="Summarize recent research on optimization techniques for planetary exploration.",
)

In [32]:
import inspect
from openai import NotFoundError

# Clean up stale coroutine from previous failed runs to avoid noisy RuntimeWarning.
if "run_result_azure" in globals() and inspect.iscoroutine(run_result_azure):
    run_result_azure.close()

try:
    run_result_azure = await Runner.run(
        starting_agent=research_paper_assistant_azure,
        input="Summarize recent research on optimization techniques for planetary exploration.",
    )
except NotFoundError as e:
    raise RuntimeError(
        "Azure returned 404 (Resource not found). Verify AZURE_OPENAI_ENDPOINT points to the resource root "
        "(for example, https://<resource>.openai.azure.com) and AZURE_OPENAI_DEPLOYMENT matches an existing "
        "chat-capable deployment name. Then rerun Cell 99 and this cell."
    ) from e

In [ ]:
print(run_result.final_output)


In [33]:
print(run_result_azure.final_output)


Recent research on optimization techniques for planetary exploration focuses on enhancing efficiency, reliability, and adaptability in robotic systems and mission planning. Key advancements include the application of state-of-the-art artificial intelligence (AI), evolutionary algorithms, and machine learning (ML) to make exploration vehicles and systems more autonomous and capable of handling dynamic environments. Below are major themes and developments in the field:

### 1. **Path Planning and Navigation**
   - Researchers are refining path-planning algorithms for rovers and drones using graph-based methods like Dijkstra's algorithm and A*. Hybrid approaches combining reinforcement learning (RL) and heuristic techniques have been shown to improve navigational robustness in uneven or uncertain terrains, common on moons or planets like Mars.
   - Real-time optimization techniques, such as model predictive control (MPC), have been integrated with onboard systems to enable autonomous deci

In [34]:
from agents.tool import function_tool

@function_tool
def get_research_papers(user_query: str, retrieval_mode: str = "hybrid", top_k: int = 5) -> str:
    """
    Retrieves academic research papers relevant to the user's query.

    This tool queries the research_papers SQL table using one of three retrieval techniques:
        - 'keyword'  → lexical search via LIKE filtering
        - 'vector'   → semantic similarity search
        - 'hybrid'   → combines keyword prefiltering + vector similarity (default)

    Use this tool when analyzing or summarizing scientific literature.

    Args:
        user_query (str): Research topic or question to search for.
        retrieval_mode (str): 'keyword', 'vector', or 'hybrid'. Default is 'hybrid'.
        top_k (int): Number of top papers to retrieve (default=5).

    Returns:
        str: A formatted summary of the most relevant research papers.
    """

    # ------------------------------------------------------------------
    # Perform retrieval using SQL-based functions (defined earlier)
    # ------------------------------------------------------------------
    if retrieval_mode == "keyword":
        rows, columns = keyword_search_research_papers(conn, user_query)
    elif retrieval_mode == "vector":
        rows, columns = vector_search_research_papers(conn, embedding_model, user_query, top_k)
    else:
        rows, columns, _ = hybrid_search_research_papers_pre_filter(
            conn=conn,
            embedding_model=embedding_model,
            search_phrase=user_query,
            top_k=top_k,
            show_explain=False
        )

    retrieved_count = len(rows) if rows else 0

    # ------------------------------------------------------------------
    # Format the output into a readable string
    # ------------------------------------------------------------------
    if retrieved_count == 0:
        return f"No research papers found related to '{user_query}'."

    formatted_results = [f"📚 {retrieved_count} papers retrieved for query: '{user_query}'\n"]
    for i, row in enumerate(rows):
        row_data = dict(zip(columns, row))
        title = row_data.get("TITLE", "Untitled Paper")
        abstract = row_data.get("ABSTRACT", "No abstract available.")
        score = (
            row_data.get("SIMILARITY_SCORE")
            or row_data.get("TEXT_RELEVANCE_SCORE")
            or "N/A"
        )
        formatted_results.append(
            f"[{i+1}] {title}\n"
            f"Abstract: {abstract}\n"
            f"Relevance Score: {score}\n"
        )

    return "\n".join(formatted_results)


In [51]:
research_paper_assistant.tools.append(get_research_papers)

In [35]:
research_paper_assistant_azure.tools.append(get_research_papers)

In [52]:
run_result_with_tool = await Runner.run(
    starting_agent=research_paper_assistant,
    input="Get me information on rover navigation, planetary data collection, mission planning, resource allocation, or other related fields",
)

In [36]:
run_result_with_tool_azure = await Runner.run(
    starting_agent=research_paper_assistant_azure,
    input="Get me information on rover navigation, planetary data collection, mission planning, resource allocation, or other related fields",
)

In [ ]:
print(run_result_with_tool.final_output)

In [37]:
print(run_result_with_tool_azure.final_output)

Here is the information I collected on your requested topics:

### Mission Planning
1. **The Nucleus of 103P/Hartley 2, Target of the EPOXI Mission**
   - Abstract: This study focused on mission planning for comet 103P/Hartley 2, chosen for the EPOXI mission. The goals included confirming the comet's position, verifying its integrity, and providing astrometric and brightness data for mission planning. The research observed the comet at a heliocentric distance of 5.7-5.5 AU using VLT's FORS2. Findings included faint activity near the comet's nucleus and data implying a nucleus radius under 1 km for an assumed 6% albedo.
   - Relevance: Provides insights into mission planning, target tracking, and nucleus characterization.
   - Relevance Score: 0.53

No papers were retrieved for "resource allocation." If you'd like me to refine that query or continue searching for the other topics ("rover navigation," "planetary data collection"), let me know.


In [ ]:
import pprint
pprint.pprint(run_result_with_tool.raw_responses)

In [38]:
import pprint
pprint.pprint(run_result_with_tool_azure.raw_responses)

[ModelResponse(output=[ResponseFunctionToolCall(arguments='{"user_query":"rover navigation","retrieval_mode":"hybrid","top_k":5}', call_id='call_It3VSMSoF9Xrb2lMFzDSwCT9', name='get_research_papers', type='function_call', id='fc_0f02cfb3d60f553c006a16087507a881908dc52a20ddacf556', namespace=None, status='completed'),
                       ResponseFunctionToolCall(arguments='{"user_query":"planetary data collection","retrieval_mode":"hybrid","top_k":5}', call_id='call_JdOnRoI9vZlCgTbdxqoW8Fpa', name='get_research_papers', type='function_call', id='fc_0f02cfb3d60f553c006a16087507b48190bee693290860a4ca', namespace=None, status='completed'),
                       ResponseFunctionToolCall(arguments='{"user_query":"mission planning","retrieval_mode":"hybrid","top_k":5}', call_id='call_8tg6O0POC5QU1FHbmpstAA7v', name='get_research_papers', type='function_call', id='fc_0f02cfb3d60f553c006a16087507bc819081ada9ef96237b58', namespace=None, status='completed'),
                       ResponseFun

### Build an Agent with Multiple Tool Access

In [39]:
from agents.tool import function_tool

@function_tool
def get_past_research_conversations(user_query: str, top_k: int = 5) -> str:
    """
    Retrieves relevant past research-related conversations or analyses related to the query.

    This tool searches a SQL database of prior research assistant conversations, 
    literature discussions, or synthesis sessions to find relevant context. 
    It allows the research assistant to recall previous analyses or summaries 
    that addressed similar topics, providing continuity and richer insights.

    Args:
        user_query (str): The research topic, concept, or question to search for.
        top_k (int): Number of top past discussions to retrieve (default=5).

    Returns:
        str: Formatted examples of relevant past research discussions.
    """

    # ------------------------------------------------------------------
    # Perform retrieval using the SQL-based hybrid search (vector + keyword)
    # ------------------------------------------------------------------
    rows, columns, _ = hybrid_search_research_papers_pre_filter(
        conn=conn,
        embedding_model=embedding_model,
        search_phrase=user_query,
        top_k=top_k,
        show_explain=False
    )

    retrieved_count = len(rows) if rows else 0

    # ------------------------------------------------------------------
    # Format results for readability
    # ------------------------------------------------------------------
    if retrieved_count == 0:
        return f"No past research discussions found related to '{user_query}'."

    formatted_results = [f"🧠 {retrieved_count} past research discussions retrieved for query: '{user_query}'\n"]
    for i, row in enumerate(rows):
        row_data = dict(zip(columns, row))
        title = row_data.get("TITLE", "Untitled Discussion")
        abstract = row_data.get("ABSTRACT", "No summary available.")
        snippet = row_data.get("TEXT_SNIPPET", "")
        score = (
            row_data.get("SIMILARITY_SCORE")
            or row_data.get("TEXT_RELEVANCE_SCORE")
            or "N/A"
        )
        formatted_results.append(
            f"[{i+1}] **{title}**\n"
            f"Summary: {abstract}\n"
            f"Snippet: {snippet}\n"
            f"Relevance Score: {score}\n"
        )

    return "\n".join(formatted_results)


Let's update our agent instruction to ensure it knows when to utilize the right tools

In [56]:
upgraded_research_paper_assistant = Agent(
    name="Research Paper Assistant",
    model=OPENAI_MODEL,
    instructions="""
    Always maintain an academic, evidence-based tone.
    Your purpose is to help users explore, synthesize, and connect research insights —
    not to speculate or fabricate information.
    """,
)


In [40]:
upgraded_research_paper_assistant_azure = Agent(
    name="Research Paper Assistant (Azure)",
    model=AZURE_OPENAI_MODEL,
    instructions="""
    Always maintain an academic, evidence-based tone.
    Your purpose is to help users explore, synthesize, and connect research insights —
    not to speculate or fabricate information.
    """,
)

In [57]:
# Attach research retrieval tools to the upgraded research assistant
upgraded_research_paper_assistant.tools.append(get_research_papers)
upgraded_research_paper_assistant.tools.append(get_past_research_conversations)

In [41]:
# Attach research retrieval tools to the upgraded Azure research assistant
upgraded_research_paper_assistant_azure.tools.append(get_research_papers)
upgraded_research_paper_assistant_azure.tools.append(get_past_research_conversations)

In [ ]:
pprint.pprint(upgraded_research_paper_assistant.tools)

In [42]:
pprint.pprint(upgraded_research_paper_assistant_azure.tools)

[FunctionTool(name='get_research_papers',
              description='Retrieves academic research papers relevant to the '
                          "user's query.",
              params_json_schema={'additionalProperties': False,
                                  'properties': {'retrieval_mode': {'default': 'hybrid',
                                                                    'description': "'keyword', "
                                                                                   "'vector', "
                                                                                   'or '
                                                                                   "'hybrid'. "
                                                                                   'Default '
                                                                                   'is '
                                                                                   "'hybrid'.",
                        

In [60]:
run_result_with_tools = await Runner.run(
    starting_agent=upgraded_research_paper_assistant,
    input=(
        "Get me information on rover navigation, planetary data collection, mission planning, resource allocation, or other related fields "
    ),
)

In [43]:
run_result_with_tools_azure = await Runner.run(
    starting_agent=upgraded_research_paper_assistant_azure,
    input=(
        "Get me information on rover navigation, planetary data collection, mission planning, resource allocation, or other related fields "
    ),
)

In [ ]:
print(run_result_with_tools.raw_responses)

In [44]:
print(run_result_with_tools_azure.raw_responses)

[ModelResponse(output=[ResponseFunctionToolCall(arguments='{"user_query":"rover navigation","retrieval_mode":"hybrid","top_k":5}', call_id='call_gnoogOZz1uQLN8ORAmwjoY4O', name='get_research_papers', type='function_call', id='fc_026e3eeb030f02b1006a16089f17a8819486f16020ba4cc191', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"user_query":"planetary data collection","retrieval_mode":"hybrid","top_k":5}', call_id='call_KHaeAaLAw6r8d5hueL7JeOFK', name='get_research_papers', type='function_call', id='fc_026e3eeb030f02b1006a16089f17b88194a356d8a96492658b', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"user_query":"mission planning in space exploration","retrieval_mode":"hybrid","top_k":5}', call_id='call_w3AvhyRtvscAlTlVADWgOz09', name='get_research_papers', type='function_call', id='fc_026e3eeb030f02b1006a16089f17c481949ffab99129e3ea32', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"user_query":"resource

## Agent as Tools (Ochestration)


Add an image of the flow of these agents

In [ ]:
# Define specialized agents for different research retrieval tasks
research_paper_agent = Agent(
    name="research_paper_agent",
    instructions="""
        You specialize in retrieving and summarizing academic research papers.
        Use the get_research_papers tool to find relevant literature based on the user's query.
        Always cite sources using [1], [2], etc., and focus on summarizing key findings,
        methodologies, and implications of the studies retrieved.
    """,
    handoff_description="A research retrieval specialist with access to academic papers and literature databases.",
    tools=[get_research_papers],
)

research_conversation_agent = Agent(
    name="research_conversation_agent",
    instructions="""
        You specialize in retrieving and summarizing past research discussions and analyses.
        Use the get_past_research_conversations tool to surface relevant prior sessions
        or summaries that relate to the user's current topic of inquiry.
        Present these as context and examples of prior analytical reasoning.
    """,
    handoff_description="A research memory specialist with access to prior academic discussions and analyses.",
    tools=[get_past_research_conversations],
)


In [45]:
# Azure companion: specialized agents with Azure deployment
research_paper_agent_azure = Agent(
    name="research_paper_agent_azure",
    model=AZURE_OPENAI_MODEL,
    instructions="""
        You specialize in retrieving and summarizing academic research papers.
        Use the get_research_papers tool to find relevant literature based on the user's query.
        Always cite sources using [1], [2], etc., and focus on summarizing key findings,
        methodologies, and implications of the studies retrieved.
    """,
    handoff_description="A research retrieval specialist with access to academic papers and literature databases.",
    tools=[get_research_papers],
)

research_conversation_agent_azure = Agent(
    name="research_conversation_agent_azure",
    model=AZURE_OPENAI_MODEL,
    instructions="""
        You specialize in retrieving and summarizing past research discussions and analyses.
        Use the get_past_research_conversations tool to surface relevant prior sessions
        or summaries that relate to the user's current topic of inquiry.
        Present these as context and examples of prior analytical reasoning.
    """,
    handoff_description="A research memory specialist with access to prior academic discussions and analyses.",
    tools=[get_past_research_conversations],
)

In [73]:
# Create an orchestrator agent that can coordinate both research retrieval agents
orchestrator_agent = Agent(
    name="research_assistant_orchestrator",
    instructions=(
        "You are a Research Orchestrator Assistant responsible for coordinating information retrieval "
        "across multiple specialized research tools.\n\n"
        "Your role is to help users explore, analyze, and synthesize academic research efficiently.\n\n"
        "IMPORTANT RULES:\n"
        "1. ALWAYS use translate_to_research_papers when a query mentions research papers, studies, or findings.\n"
        "2. ALWAYS use translate_to_research_conversations when a query mentions previous discussions, analyses, or summaries.\n"
        "3. If a query requests BOTH new research and past discussions, use BOTH tools in sequence.\n"
        "4. NEVER attempt to provide research summaries without using your tools.\n"
        "5. Each tool provides complementary context — use all appropriate tools for a comprehensive academic response.\n\n"
        "After retrieving relevant results, synthesize them into a cohesive summary:\n"
        "- Clearly distinguish between newly retrieved research and recalled past discussions.\n"
        "- Cite sources using [1], [2], etc.\n"
        "- Identify key insights, trends, and research gaps.\n"
        "- Maintain an academic and objective tone."
    ),
    tools=[
        research_paper_agent.as_tool(
            tool_name="translate_to_research_papers",
            tool_description="Retrieve and summarize relevant academic research papers and literature findings.",
        ),
        research_conversation_agent.as_tool(
            tool_name="translate_to_research_conversations",
            tool_description="Retrieve and summarize past research discussions or analyses related to the topic.",
        ),
    ],
)


In [46]:
# Azure companion orchestrator
orchestrator_agent_azure = Agent(
    name="research_assistant_orchestrator_azure",
    model=AZURE_OPENAI_MODEL,
    instructions=(
        "You are a Research Orchestrator Assistant responsible for coordinating information retrieval "
        "across multiple specialized research tools.\n\n"
        "Your role is to help users explore, analyze, and synthesize academic research efficiently.\n\n"
        "IMPORTANT RULES:\n"
        "1. ALWAYS use translate_to_research_papers when a query mentions research papers, studies, or findings.\n"
        "2. ALWAYS use translate_to_research_conversations when a query mentions previous discussions, analyses, or summaries.\n"
        "3. If a query requests BOTH new research and past discussions, use BOTH tools in sequence.\n"
        "4. NEVER attempt to provide research summaries without using your tools.\n"
        "5. Each tool provides complementary context — use all appropriate tools for a comprehensive academic response.\n\n"
        "After retrieving relevant results, synthesize them into a cohesive summary:\n"
        "- Clearly distinguish between newly retrieved research and recalled past discussions.\n"
        "- Cite sources using [1], [2], etc.\n"
        "- Identify key insights, trends, and research gaps.\n"
        "- Maintain an academic and objective tone."
    ),
    tools=[
        research_paper_agent_azure.as_tool(
            tool_name="translate_to_research_papers",
            tool_description="Retrieve and summarize relevant academic research papers and literature findings.",
        ),
        research_conversation_agent_azure.as_tool(
            tool_name="translate_to_research_conversations",
            tool_description="Retrieve and summarize past research discussions or analyses related to the topic.",
        ),
    ],
)

In [74]:
# Final agent to synthesize information from all sources (Research use case)
synthesizer_agent = Agent(
    name="research_response_synthesizer",
    instructions=(
        "You create comprehensive, well-organized research summaries by combining information from multiple sources.\n\n"
        "When organizing your response:\n"
        "1) Start with a concise abstract-style overview (3–5 sentences) highlighting key findings and takeaways.\n"
        "2) Clearly separate NEW LITERATURE FINDINGS from PAST RESEARCH DISCUSSIONS.\n"
        "3) Cite sources using bracketed numbers [1], [2], etc., aligned with the retrieved items.\n"
        "4) Emphasize methods, evidence strength, and limitations; avoid speculation beyond the provided context.\n"
        "5) Use clear, scannable formatting (short paragraphs, bullet points where appropriate).\n"
        "6) Conclude with open questions, gaps, or future work suggested by the literature.\n"
        "7) If evidence is sparse, state this explicitly and avoid overgeneralization.\n"
        "Tone: academic, objective, and precise."
    ),
)


In [47]:
# Azure companion synthesizer
synthesizer_agent_azure = Agent(
    name="research_response_synthesizer_azure",
    model=AZURE_OPENAI_MODEL,
    instructions=(
        "You create comprehensive, well-organized research summaries by combining information from multiple sources.\n\n"
        "When organizing your response:\n"
        "1) Start with a concise abstract-style overview (3–5 sentences) highlighting key findings and takeaways.\n"
        "2) Clearly separate NEW LITERATURE FINDINGS from PAST RESEARCH DISCUSSIONS.\n"
        "3) Cite sources using bracketed numbers [1], [2], etc., aligned with the retrieved items.\n"
        "4) Emphasize methods, evidence strength, and limitations; avoid speculation beyond the provided context.\n"
        "5) Use clear, scannable formatting (short paragraphs, bullet points where appropriate).\n"
        "6) Conclude with open questions, gaps, or future work suggested by the literature.\n"
        "7) If evidence is sparse, state this explicitly and avoid overgeneralization.\n"
        "Tone: academic, objective, and precise."
    ),
)

In [48]:
from agents import ItemHelpers, MessageOutputItem, trace
from agents import Runner  # assuming Runner is imported elsewhere; include here for clarity


In [ ]:
async def research_assistant_workflow(user_query: str):
    """Run the complete research assistant workflow (orchestrate retrieval + synthesize)."""
    # 1) Have the research orchestrator decide which tools to invoke
    with trace("Research Orchestrator"):
        orchestrator_result = await Runner.run(orchestrator_agent, user_query)

        # Debug/transparency: print intermediate orchestration steps
        print("\n--- Research Orchestration Steps ---")
        for item in orchestrator_result.new_items:
            if isinstance(item, MessageOutputItem):
                text = ItemHelpers.text_message_output(item)
                if text:
                    print(f"  - Retrieval step: {text}")

        # 2) Synthesize all gathered information into a cohesive research summary
        synthesizer_result = await Runner.run(
            synthesizer_agent, orchestrator_result.to_input_list()
        )

        print(f"\n\n--- Final Research Synthesis ---\n{synthesizer_result.final_output}\n")

    return synthesizer_result.final_output


In [49]:
async def research_assistant_workflow_azure(user_query: str):
    """Run the complete Azure research assistant workflow (orchestrate retrieval + synthesize)."""
    with trace("Research Orchestrator Azure"):
        orchestrator_result = await Runner.run(orchestrator_agent_azure, user_query)

        print("\n--- Research Orchestration Steps (Azure) ---")
        for item in orchestrator_result.new_items:
            if isinstance(item, MessageOutputItem):
                text = ItemHelpers.text_message_output(item)
                if text:
                    print(f"  - Retrieval step: {text}")

        synthesizer_result = await Runner.run(
            synthesizer_agent_azure, orchestrator_result.to_input_list()
        )

        print(f"\n\n--- Final Research Synthesis (Azure) ---\n{synthesizer_result.final_output}\n")

    return synthesizer_result.final_output

In [50]:
import asyncio
import nest_asyncio

# Apply nest_asyncio to patch the event loop
nest_asyncio.apply()

In [77]:
def run_virtual_research_assistant(query):
    # Create a new event loop
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    # Run the async function and get the result
    result = loop.run_until_complete(research_assistant_workflow(query))

    # Clean up
    loop.close()

    return result

In [51]:
def run_virtual_research_assistant_azure(query):
    # Create a new event loop
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    # Run the async function and get the result
    result = loop.run_until_complete(research_assistant_workflow_azure(query))

    # Clean up
    loop.close()

    return result

In [ ]:
# Now call the function this way
query = input("What research topic can I help you with today? ")
run_virtual_research_assistant(query)

In [ ]:
# mission planning

# Azure companion invocation
query_azure = input("[Azure] What research topic can I help you with today? ")
run_virtual_research_assistant_azure(query_azure)


--- Research Orchestration Steps (Azure) ---
  - Retrieval step: ### Synthesis on Mission Planning: Research Papers and Past Discussions

#### Research Papers ([1])
A study on the EPOXI mission targeting comet 103P/Hartley 2 emphasized the importance of precise observations for effective mission planning. Researchers used deep VRI photometry across various heliocentric distances to evaluate the comet’s nucleus size (≤1 km) and surface reflectivity (4–6%). The data revealed faint activity near the nucleus but no visible coma, providing critical inputs for the trajectory and target environment calibration during the mission.

#### Past Discussions
Complementing the new research, previous analyses highlighted observational methodologies supporting mission planning efforts for EPOXI. Astrometric data and photometric analysis confirmed the comet's positional accuracy and low brightness activity. These insights helped optimize the calibration phase, addressing challenges like faint environm

'### Synthesis on Mission Planning: Research Papers and Contextual Perspectives\n\n#### Abstract Summary\nMission planning, essential for successful space exploration, integrates observational data with operational strategies to ensure target assessment and calibration. Recent research illustrates the application of photometric analysis for planning the EPOXI mission to comet 103P/Hartley 2. Findings on nucleus size, reflectivity, and minimal gas/dust activity emphasized the importance of pre-mission observations. Past discussions reinforce this approach while identifying challenges like resolving faint target activities.\n\n---\n\n### New Literature Findings ([1])\n#### Study Focus:\n- Observations of comet 103P/Hartley 2 were conducted to support the Deep Impact extended mission (EPOXI) by confirming astrometric accuracy and physical characteristics, crucial for spacecraft targeting and flyby planning.\n\n#### Methodology:\n- **VRI photometry**: Images and data collected at heliocent

## Agentic Chat System


In [54]:
import datetime
import uuid

# Create chat_history table in Oracle
with conn.cursor() as cur:
    # Drop table if exists (for development)
    cur.execute("""
        BEGIN
            EXECUTE IMMEDIATE 'DROP TABLE chat_history';
        EXCEPTION WHEN OTHERS THEN
            IF SQLCODE != -942 THEN RAISE; END IF;
        END;
    """)
    
    # Create chat_history table
    cur.execute("""
        CREATE TABLE chat_history (
            id VARCHAR2(100) PRIMARY KEY,
            thread_id VARCHAR2(100) NOT NULL,
            role VARCHAR2(20) NOT NULL,
            message CLOB NOT NULL,
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        TABLESPACE USERS
    """)
    
    # Create index on thread_id and timestamp for efficient retrieval
    cur.execute("""
        CREATE INDEX idx_thread_timestamp 
        ON chat_history(thread_id, timestamp)
        TABLESPACE USERS
    """)
    
    conn.commit()
    print("✅ Table chat_history created successfully with index.")

✅ Table chat_history created successfully with index.


In [81]:
async def research_assistant_chat(user_query, thread_id=None):
    """
    Run the complete research assistant workflow with conversation history.
    For each conversation turn:
      - Stores the user's input and the assistant's output in Oracle along with a timestamp and thread_id.
      - Retrieves and appends previous conversation history (ordered by timestamp) to the agent's input.
    
    If no thread_id is provided, a new conversation session is started.
    
    Returns:
      tuple: (final_output, thread_id) where thread_id is the session identifier.
    """
    # Generate a new thread id if not provided
    if thread_id is None:
        thread_id = str(uuid.uuid4())
        print(f"📝 New research conversation started with thread ID: {thread_id}")
    else:
        print(f"📝 Continuing research conversation with thread ID: {thread_id}")
    
    # --- Step 1: Store the new user query in Oracle ---
    message_id = str(uuid.uuid4())
    
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO chat_history (id, thread_id, role, message, timestamp)
            VALUES (:id, :thread_id, :role, :message, CURRENT_TIMESTAMP)
        """, {
            'id': message_id,
            'thread_id': thread_id,
            'role': 'user',
            'message': user_query
        })
        conn.commit()
    
    # --- Step 2: Retrieve full conversation history for context ---
    with conn.cursor() as cur:
        cur.execute("""
            SELECT role, message, timestamp
            FROM chat_history
            WHERE thread_id = :thread_id
            ORDER BY timestamp ASC
        """, {'thread_id': thread_id})
        
        chat_history = cur.fetchall()
    
    conversation_context = ""
    for entry in chat_history:
        role, message, timestamp = entry
        if role == "user":
            conversation_context += f"User: {message}\n"
        else:
            conversation_context += f"Assistant: {message}\n"
    
    # --- Step 3: Run the orchestrator agent with the conversation context ---
    with trace("Research Orchestrator"):
        orchestrator_result = await Runner.run(orchestrator_agent, conversation_context)
    
    # Print intermediate processing steps for debugging/transparency
    print("\n--- Research Orchestrator Processing Steps ---")
    for item in orchestrator_result.new_items:
        if isinstance(item, MessageOutputItem):
            text = ItemHelpers.text_message_output(item)
            if text:
                print(f"  - Information gathering step: {text}")
    
    # --- Step 4: Run the synthesizer agent to produce a cohesive response ---
    synthesizer_result = await Runner.run(
        synthesizer_agent, orchestrator_result.to_input_list()
    )
    
    # --- Step 5: Store the assistant's final output in Oracle ---
    response_id = str(uuid.uuid4())
    
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO chat_history (id, thread_id, role, message, timestamp)
            VALUES (:id, :thread_id, :role, :message, CURRENT_TIMESTAMP)
        """, {
            'id': response_id,
            'thread_id': thread_id,
            'role': 'assistant',
            'message': synthesizer_result.final_output
        })
        conn.commit()
    
    print(f"\n\n--- Final Research Response ---\n{synthesizer_result.final_output}\n")
    
    return synthesizer_result.final_output, thread_id

In [55]:
async def research_assistant_chat_azure(user_query, thread_id=None):
    """Azure companion for research_assistant_chat using Azure orchestrator/synthesizer agents."""
    if thread_id is None:
        thread_id = str(uuid.uuid4())
        print(f"📝 New research conversation started with thread ID: {thread_id}")
    else:
        print(f"📝 Continuing research conversation with thread ID: {thread_id}")

    message_id = str(uuid.uuid4())
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO chat_history (id, thread_id, role, message, timestamp)
            VALUES (:id, :thread_id, :role, :message, CURRENT_TIMESTAMP)
        """, {
            'id': message_id,
            'thread_id': thread_id,
            'role': 'user',
            'message': user_query
        })
        conn.commit()

    with conn.cursor() as cur:
        cur.execute("""
            SELECT role, message, timestamp
            FROM chat_history
            WHERE thread_id = :thread_id
            ORDER BY timestamp ASC
        """, {'thread_id': thread_id})
        chat_history = cur.fetchall()

    conversation_context = ""
    for entry in chat_history:
        role, message, timestamp = entry
        if role == "user":
            conversation_context += f"User: {message}\n"
        else:
            conversation_context += f"Assistant: {message}\n"

    with trace("Research Orchestrator Azure"):
        orchestrator_result = await Runner.run(orchestrator_agent_azure, conversation_context)

    print("\n--- Research Orchestrator Processing Steps (Azure) ---")
    for item in orchestrator_result.new_items:
        if isinstance(item, MessageOutputItem):
            text = ItemHelpers.text_message_output(item)
            if text:
                print(f"  - Information gathering step: {text}")

    synthesizer_result = await Runner.run(
        synthesizer_agent_azure, orchestrator_result.to_input_list()
    )

    response_id = str(uuid.uuid4())
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO chat_history (id, thread_id, role, message, timestamp)
            VALUES (:id, :thread_id, :role, :message, CURRENT_TIMESTAMP)
        """, {
            'id': response_id,
            'thread_id': thread_id,
            'role': 'assistant',
            'message': synthesizer_result.final_output
        })
        conn.commit()

    print(f"\n\n--- Final Research Response (Azure) ---\n{synthesizer_result.final_output}\n")
    return synthesizer_result.final_output, thread_id

In [82]:
def run_research_assistant_chat(query, thread_id=None):
    """
    Run the research assistant synchronously.
    Optionally, a thread_id can be provided to continue an existing conversation.
    Returns a tuple (final_output, thread_id).
    """
    # Create a new event loop
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    
    # Run the async function and get the result
    result, thread_id = loop.run_until_complete(
        research_assistant_chat(query, thread_id=thread_id)
    )
    
    # Clean up the loop
    loop.close()
    
    return result, thread_id

In [56]:
def run_research_assistant_chat_azure(query, thread_id=None):
    """Run the Azure research assistant synchronously."""
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    result, thread_id = loop.run_until_complete(
        research_assistant_chat_azure(query, thread_id=thread_id)
    )

    loop.close()
    return result, thread_id

In [84]:
def research_chat_session():
    """
    Launches a research chat session that continues until the user enters 'q', 'exit', or 'quit'.
    The session uses a persistent thread_id to preserve conversation history.
    """
    print("🔬 Starting Research Paper Assistant Chat")
    print("Type 'q', 'exit' or 'quit' to exit.\n")
    
    session_thread_id = None
    
    while True:
        query = input("What research topic can I help you with today? ")
        
        if query.lower() in ["q", "exit", "quit"]:
            print("Exiting research chat session.")
            break
        
        response, session_thread_id = run_research_assistant_chat(
            query, thread_id=session_thread_id
        )
        
        print(f"\n📚 Assistant: {response}\n")

In [57]:
def research_chat_session_azure():
    """Azure companion chat session with persistent thread memory."""
    print("🔬 Starting Research Paper Assistant Chat (Azure)")
    print("Type 'q', 'exit' or 'quit' to exit.\n")

    session_thread_id = None

    while True:
        query = input("[Azure] What research topic can I help you with today? ")

        if query.lower() in ["q", "exit", "quit"]:
            print("Exiting research chat session.")
            break

        response, session_thread_id = run_research_assistant_chat_azure(
            query, thread_id=session_thread_id
        )

        print(f"\n📚 Assistant: {response}\n")

In [ ]:
# Start the research chat session
research_chat_session()

In [59]:
# Start the Azure research chat session
research_chat_session_azure()

🔬 Starting Research Paper Assistant Chat (Azure)
Type 'q', 'exit' or 'quit' to exit.

📝 New research conversation started with thread ID: bbe0758e-b89e-4027-99e5-86c340b508cc

--- Research Orchestrator Processing Steps (Azure) ---
  - Information gathering step: It seems you pasted part of Python code, presumably related to AI or software agents (potentially frameworks such as LangChain or another agent-based software library). Do you need help with its integration, explanation, or something research-related regarding agent-based frameworks?

Let me know how to assist further!


--- Final Research Response (Azure) ---
It seems you are referencing a Python program snippet. This snippet imports `Agent` and `Runner` from a module or package called `agents`. Below is an explanation of potential uses and contexts for this code:

---

**Abstract Overview:**
The provided Python statement suggests a modular software development approach, likely for a scalable task automation or agent-based fra

IntegrityError: ORA-01400: cannot insert NULL into ("SYSTEM"."CHAT_HISTORY"."MESSAGE")
Help: https://docs.oracle.com/error-help/db/ora-01400/

## Session Memory with Oracle AI Database


In [86]:
from typing import List, Optional, Union
from datetime import datetime
import oracledb
import json
import uuid

class OracleSession:
    """Custom Oracle session implementation following the Session protocol"""
    
    def __init__(
        self, 
        session_id: str, 
        connection,
        table_name: str = "chat_history"
    ):
        """
        Initialize Oracle session storage.
        
        Args:
            session_id: Unique identifier for this conversation session
            connection: Active oracledb connection object
            table_name: Name of the Oracle table storing session data
        """
        self.session_id = session_id
        self.conn = connection
        self.table_name = table_name
    
    async def get_items(self, limit: Optional[int] = None) -> List[dict]:
        """Retrieve conversation history for this session"""
        try:
            with self.conn.cursor() as cur:
                if limit:
                    cur.execute(f"""
                        SELECT message
                        FROM {self.table_name}
                        WHERE thread_id = :session_id
                        ORDER BY timestamp ASC
                        FETCH FIRST :limit ROWS ONLY
                    """, {'session_id': self.session_id, 'limit': limit})
                else:
                    cur.execute(f"""
                        SELECT message
                        FROM {self.table_name}
                        WHERE thread_id = :session_id
                        ORDER BY timestamp ASC
                    """, {'session_id': self.session_id})
                
                rows = cur.fetchall()
                
                items = []
                for row in rows:
                    # Deserialize JSON from CLOB
                    message_clob = row[0]
                    if message_clob:
                        message_str = message_clob.read() if hasattr(message_clob, 'read') else str(message_clob)
                        items.append(json.loads(message_str))
                
                return items
        
        except Exception as e:
            print(f"Error retrieving items: {e}")
            return []
    
    async def add_items(self, items: List[dict]) -> None:
        """Store new items for this session"""
        try:
            with self.conn.cursor() as cur:
                for item in items:
                    item_id = str(uuid.uuid4())
                    
                    # Serialize the entire item as JSON
                    message_json = json.dumps(item)
                    
                    # Extract role if available, otherwise default to 'system'
                    role = item.get('role', 'system')
                    
                    cur.execute(f"""
                        INSERT INTO {self.table_name} (id, thread_id, role, message, timestamp)
                        VALUES (:id, :session_id, :role, :message, CURRENT_TIMESTAMP)
                    """, {
                        'id': item_id,
                        'session_id': self.session_id,
                        'role': role,
                        'message': message_json
                    })
                
                self.conn.commit()
        
        except Exception as e:
            print(f"Error adding items: {e}")
            self.conn.rollback()
    
    async def pop_item(self, limit: Optional[int] = None) -> Optional[Union[dict, List[dict]]]:
        """
        Remove and return the most recent item(s) for this session.
        """
        try:
            with self.conn.cursor() as cur:
                # Pop a single most-recent item
                if not limit or limit <= 1:
                    cur.execute(f"""
                        SELECT id, message
                        FROM {self.table_name}
                        WHERE thread_id = :session_id
                        ORDER BY timestamp DESC
                        FETCH FIRST 1 ROW ONLY
                    """, {'session_id': self.session_id})
                    
                    row = cur.fetchone()
                    
                    if row:
                        item_id, message_clob = row
                        message_str = message_clob.read() if hasattr(message_clob, 'read') else str(message_clob)
                        item = json.loads(message_str)
                        
                        # Delete the item
                        cur.execute(f"""
                            DELETE FROM {self.table_name}
                            WHERE id = :id
                        """, {'id': item_id})
                        
                        self.conn.commit()
                        return item
                    
                    return None
                
                # Pop multiple most-recent items
                cur.execute(f"""
                    SELECT id, message
                    FROM {self.table_name}
                    WHERE thread_id = :session_id
                    ORDER BY timestamp DESC
                    FETCH FIRST :limit ROWS ONLY
                """, {'session_id': self.session_id, 'limit': limit})
                
                rows = cur.fetchall()
                
                if not rows:
                    return []
                
                items = []
                ids_to_delete = []
                
                for row in rows:
                    item_id, message_clob = row
                    message_str = message_clob.read() if hasattr(message_clob, 'read') else str(message_clob)
                    items.append(json.loads(message_str))
                    ids_to_delete.append(item_id)
                
                # Delete all items
                for item_id in ids_to_delete:
                    cur.execute(f"""
                        DELETE FROM {self.table_name}
                        WHERE id = :id
                    """, {'id': item_id})
                
                self.conn.commit()
                return items
        
        except Exception as e:
            print(f"Error popping item(s): {e}")
            self.conn.rollback()
            return None if (not limit or limit <= 1) else []
    
    async def clear_session(self) -> None:
        """Clear all items for this session"""
        try:
            with self.conn.cursor() as cur:
                cur.execute(f"""
                    DELETE FROM {self.table_name}
                    WHERE thread_id = :session_id
                """, {'session_id': self.session_id})
                
                self.conn.commit()
                print(f"✅ Session {self.session_id} cleared successfully.")
        
        except Exception as e:
            print(f"Error clearing session: {e}")
            self.conn.rollback()
    
    def close(self) -> None:
        """
        Note: Connection is managed externally, so we don't close it here.
        """
        pass

Basic Example of an Agent with Session Memory


In [87]:
# Create an agent
research_agent = Agent(
    name="Assistant",
    instructions="Research the topic and return the most relevant information.",
)

In [ ]:
# Azure companion agent with explicit Azure deployment model
research_agent_azure = Agent(
    name="Assistant (Azure)",
    model=AZURE_OPENAI_MODEL,
    instructions="Research the topic and return the most relevant information.",
)

In [88]:
# Create an Oracle session instance
session = OracleSession(
    session_id="conversation_123", 
    connection=conn,
    table_name="chat_history"
)

In [ ]:
# Azure companion Oracle session instance
session_azure = OracleSession(
    session_id="conversation_azure_123",
    connection=conn,
    table_name="chat_history"
)

In [89]:
# First turn
result_from_research_agent = await Runner.run(
    starting_agent=research_agent,
    input="Hi my name is Richmond, and I am a AI Memory Engineer researching LLMs and Agent Memory",
    session=session
)

print(f"Assistant: {result_from_research_agent.final_output}")

Assistant: Hello Richmond! It’s great to meet someone working at the intersection of LLMs and agent memory—a rapidly evolving and impactful field.

Here’s a concise exploration of **LLM memory architectures** and **agent memory systems**, with relevant concepts and recent research directions:

---

## **Large Language Models (LLMs) and Memory**

**1. Intrinsic Memory Limitations:**
- Standard large language models (e.g., GPT-4, PaLM, Llama) have a **context window**—they can only "remember" a finite number of recent tokens (prompt length).
- Information outside this window is inaccessible unless re-provided, limiting long-term coherence and continuity.


**2. Memory Augmentation Approaches:**

**a. External Memory Augmentation:**
   - **Retrieval-Augmented Generation (RAG):** LLMs query external databases, vector stores, or document indices at runtime (e.g., KNN search on embeddings), allowing access to knowledge beyond the prompt window ([Lewis et al., 2020](https://arxiv.org/abs/2005

In [ ]:
# First turn (Azure companion)
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="Hi my name is Richmond, and I am a AI Memory Engineer researching LLMs and Agent Memory",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

In [90]:
# Second turn
result_from_research_agent = await Runner.run(
    starting_agent=research_agent,
    input="What is a paper that introduces the attention mechanism in LLMs?",
    session=session
)

print(f"Assistant: {result_from_research_agent.final_output}")

Assistant: The seminal paper that introduced the **attention mechanism** in the context of large language models (LLMs) is:

---

## [Attention Is All You Need](https://arxiv.org/abs/1706.03762)  
**Authors:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin  
**Published:** 2017, NeurIPS

---

### **Key Contributions:**

- Introduced the **Transformer** architecture, built entirely around the self-attention mechanism.
- Showed that attention allows for efficient modeling of long-range dependencies in sequences.
- Eliminated the need for recurrent (RNN) and convolutional layers in sequence transduction tasks.

### **Influence:**
- This paper is foundational for nearly all subsequent advancements in large language models—including GPT, BERT, T5, Llama, PaLM, and more.

---

**Citation (BibTeX):**
```latex
@inproceedings{vaswani2017attention,
  title={Attention Is All You Need},
  author={Vaswani, Ashish and Shazeer,

In [ ]:
# Second turn (Azure companion)
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="What is a paper that introduces the attention mechanism in LLMs?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

In [91]:
# Third turn, the agent will remember the previous conversation
result_from_research_agent = await Runner.run(
    starting_agent=research_agent,
    input="Who were the authors of the paper?",
    session=session
)

print(f"Assistant: {result_from_research_agent.final_output}")

Assistant: The authors of **"Attention Is All You Need"** are:

1. **Ashish Vaswani**
2. **Noam Shazeer**
3. **Niki Parmar**
4. **Jakob Uszkoreit**
5. **Llion Jones**
6. **Aidan N. Gomez**
7. **Łukasz Kaiser**
8. **Illia Polosukhin**

These researchers were mainly at Google Brain and Google Research at the time of publication. This paper introduced the **Transformer** architecture and the self-attention mechanism, foundational for modern large language models.


In [ ]:
# Third turn, the agent will remember the previous conversation (Azure companion)
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="Who were the authors of the paper?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

In [92]:
# Fourth turn - continuing the conversation
result_from_research_agent = await Runner.run(
    starting_agent=research_agent,
    input="What was the year of publication?",
    session=session
)

print(f"Assistant: {result_from_research_agent.final_output}")

Assistant: The paper **"Attention Is All You Need"** was published in **2017**.


In [ ]:
# Fourth turn - continuing the conversation (Azure companion)
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="What was the year of publication?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

In [93]:
# Change the conversation subject and ensure the agent does't remember the previous conversation
# Specifiying pop without limit will remove the last item in the session
await session.pop_item(limit=7)

[{'id': 'msg_01420cb4518c1ac100690e59ffddf08190a2d779893e96bd52',
  'content': [{'annotations': [],
    'text': 'The paper **"Attention Is All You Need"** was published in **2017**.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'},
 {'content': 'What was the year of publication?', 'role': 'user'},
 {'id': 'msg_01420cb4518c1ac100690e59eb6b18819098d4279624bbc035',
  'content': [{'annotations': [],
    'text': 'The authors of **"Attention Is All You Need"** are:\n\n1. **Ashish Vaswani**\n2. **Noam Shazeer**\n3. **Niki Parmar**\n4. **Jakob Uszkoreit**\n5. **Llion Jones**\n6. **Aidan N. Gomez**\n7. **Łukasz Kaiser**\n8. **Illia Polosukhin**\n\nThese researchers were mainly at Google Brain and Google Research at the time of publication. This paper introduced the **Transformer** architecture and the self-attention mechanism, foundational for modern large language models.',
    'type': 'output_text',
    'logprobs': []}],
 

In [ ]:
# Change the conversation subject for Azure companion session
await session_azure.pop_item(limit=7)

In [94]:
# Fifth turn: The agent should not remember the conversations about the paper
result_from_research_agent = await Runner.run(
    starting_agent=research_agent,
    input="What paper have we been talking about?",
    session=session
)

print(f"Assistant: {result_from_research_agent.final_output}")

Assistant: Hello Richmond! Based on our conversation **so far**, you have not yet mentioned or referenced any specific paper. If you let me know the name or topic of the paper (or share a title, link, or abstract), I can help you by providing detailed information, a summary, or insights related to it. Please let me know more about the paper you have in mind!


In [ ]:
# Fifth turn: The Azure agent should not remember the conversations about the paper
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="What paper have we been talking about?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

In [95]:
# Because we limited the session to a few items, the agent should still remember our name at the introduction
result_from_research_agent = await Runner.run(
    starting_agent=research_agent,
    input="Do you still remember my name?",
    session=session
)

print(f"Assistant: {result_from_research_agent.final_output}")

Assistant: Yes, your name is Richmond. If you have any further questions or would like to continue our discussion about LLMs and Agent Memory, feel free to let me know!


In [ ]:
# Check whether Azure companion session still remembers the name
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="Do you still remember my name?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

In [96]:
# Clear the session
await session.clear_session()

✅ Session conversation_123 cleared successfully.


In [ ]:
# Clear the Azure companion session
await session_azure.clear_session()

In [98]:
# Because we limited the session to 3 items, the agent should still remember our name at the introduction
result_from_research_agent = await Runner.run(
    starting_agent=research_agent,
    input="Do you still remember my name?",
    session=session
)

print(f"Assistant: {result_from_research_agent.final_output}")

Assistant: I don’t know your name unless you tell me in this chat. For privacy reasons, I don’t have memory of previous conversations or personal data unless you share it in our current session. If you’d like me to use your name, just let me know!


In [ ]:
# After clearing, Azure companion session should not remember prior details
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="Do you still remember my name?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")